# Worked hand verification of slab-weight pathways using dataset slabs

This notebook independently calculates every intermediate parameter for the
**four shear-weight pathways** and **32 shear-weight-with-elasticity pathways**
using actual slabs from the paper's dataset. It compares nominal values, measurement
standard deviations, and calculation availability with SnowPyt-MechParams.

Each pathway has its own section. We find its complete successful set, randomly
select one successful slab when available, and independently select ten slabs
without a success requirement. All selected observations come from the dataset;
a fixed seed makes the selections reproducible.

The physical quantities are weight per unit area, $W$, and its slope-parallel
component, $W_{\mathrm{shear}}$. The elasticity target returns the same shear
weight when each layer has valid Young's modulus $E$ and Poisson's ratio $\nu$.
Shear modulus and laminate stiffnesses are outside these pathways.

Kim Table 6 → Bergfeld → Köchle has incompatible grain-form domains. Other
pathways may also have empty successful sets in this observed cohort. In every
case, the ten unfiltered examples check available upstream calculations and
predict unavailable outputs independently.

Agreement verifies the numerical implementation and pathway wiring for the
selected dataset examples; it does not establish empirical accuracy under all
snow conditions.

## 1. Inputs, units, uncertainty, and pathway inventory

Layers are ordered from the surface downward. Their thicknesses, grain forms,
hand-hardness codes, grain sizes, measured densities, and the slab slope angle
come from the dataset. Each worked example displays its original observations
and measurement standard deviations. Missing or nonfinite inputs remain unavailable.

Use the stored measurement standard deviations: thickness uncertainty is 5%,
measured-density uncertainty is 10%, grain-size uncertainty is 0.5 mm, and slope
uncertainty is 2° when those observations are present. Convert each reported
hardness code using the independent table below, with $\sigma_H=0.67$.
Distinct layer measurements are independent. The one slope angle is shared by
all layers and contributes to shear-weight uncertainty only once.

All manual uncertainties use first-order analytical propagation:

$$\sigma_f^2=\sum_j\left(\frac{\partial f}{\partial x_j}\sigma_{x_j}\right)^2.$$

Match the paper by including measurement uncertainty and disabling empirical
method uncertainty (`include_method_uncertainty=False`). Coefficients, grain-form
means of $\nu$, $g=9.81$ m/s², $\rho_{\mathrm{ice}}=917$ kg/m³, and
$E_{\mathrm{ice}}=10{,}000$ MPa are exact. Domain checks use nominal values;
uncertainties crossing a domain boundary are not truncated.

Density is in kg/m³, thickness is converted from cm to m, weights are in
N/m² (Pa), $E$ is in MPa, and $\nu$ is dimensionless. Manual calculations use
ordinary floating-point arithmetic, explicit equations, and analytical derivatives.
Framework-side checks use the dataset's uncertain-number inputs.

The engine inventory below must contain four weight pathways and 32 elasticity
pathways without duplicates. Enumeration alone does not guarantee success on
the observed dataset.

In [331]:
# Run this notebook from top to bottom. A code cell can use variables created in earlier
# cells. Each pathway section selects methods and recalculates its dataset samples.
# An import makes another Python module available. Path works with file locations; sys
# controls where Python searches for imports; math supplies scalar mathematical
# functions; warnings controls warning messages.
from pathlib import Path
import sys
import math
import warnings
# Layer and Slab are dataclass objects: Python objects with named data fields. These
# utilities let the later snapshot helper inspect every field.
from dataclasses import fields, is_dataclass

# The words "as np" give NumPy a short nickname. NumPy supplies finite-number
# and numerical-agreement checks for the comparison tables.
import numpy as np
# Pandas is nicknamed pd. Its DataFrame is a table with named columns, used to collect
# and display our comparisons.
import pandas as pd
# display shows rich notebook output; Markdown renders explanatory text. These
# display tools do not calculate physical quantities.
from IPython.display import display, Markdown
# UFloat identifies the uncertain numbers stored in dataset observations. Manual
# answers use ordinary numbers and explicit uncertainty formulas.
from uncertainties import UFloat

# Locate this checkout from paper_analysis/, the repository, or its parent.
# Find the repository containing the framework code. Path.cwd() is the current working
# folder. Its parents are the folders above it. The leading * expands those parents into
# the search list.
# The expression below is a generator: it supplies candidate folders one at a time.
# next(...) takes the first candidate satisfying the final if condition. The / operator
# joins folder names when its left operand is a Path.
repo_root = next(
    candidate
    for base in [Path.cwd(), *Path.cwd().parents]
    for candidate in [base, base / "SnowPyt-MechParams"]
    if (candidate / "src" / "snowpyt_mechparams").is_dir()
)
# Put this checkout's src folder first in Python's import search list. This ensures the
# comparisons use the local framework implementation.
sys.path.insert(0, str(repo_root / "src"))
# ExecutionEngine finds and runs pathways. ExecutionConfig chooses execution settings.
from snowpyt_mechparams.execution import ExecutionConfig, ExecutionEngine

# Fixed method coefficients generate zero-uncertainty UFloat objects internally.
# Suppress only this known warning; numerical failures still raise assertions.
# A warning is a message, whereas an exception stops execution. Ignore only the known
# message about exact (zero-uncertainty) coefficients. All verification assertions
# remain active.
warnings.filterwarnings(
    "ignore", message="Using UFloat objects with std_dev==0.*", category=UserWarning
)
# Construct one engine object. Its methods are functions accessed with a dot, for
# example engine.execute_single(...).
engine = ExecutionEngine()
# This keyword argument switches off empirical fit uncertainty. Measurement uncertainty
# still propagates through the framework, matching the hand calculations.
config = ExecutionConfig(include_method_uncertainty=False)
# Assign two variables at once. 1e-10 means 10 to the power -10. These very small
# relative and absolute tolerances allow floating-point rounding differences.
RTOL, ATOL = 1e-10, 1e-12
# These uppercase variables hold the framework's exact target names. Strings are text
# enclosed in quotation marks; they identify which output to calculate.
WEIGHT = "slab_weight_shear"
ELASTICITY = "slab_weight_shear_with_elasticity"
# Lists use square brackets. Each string below identifies one available method.
# Combining one density, one E, and one ν method produces an elasticity pathway.
density_methods = ["data_flow", "geldsetzer", "kim_jamieson_table2", "kim_jamieson_table6"]
elastic_methods = ["bergfeld", "kochle", "wautier", "schottner"]
poisson_methods = ["kochle", "srivastava"]
# A dictionary uses braces and stores key: value pairs. This dictionary comprehension
# repeats the engine call for each target and stores the resulting list of pathways
# under that target's name.
# Pathway enumeration depends on graph structure. Being listed does not guarantee that a
# method combination can succeed on real inputs.
inventory = {target: engine.list_available_pathways(target) for target in [WEIGHT, ELASTICITY]}

# def introduces a reusable function; its indented lines are its body. This function
# converts a method dictionary into a fixed-order tuple (density method, E method, ν
# method). A tuple is an ordered collection written with parentheses. It can serve as a
# dictionary key or set member.
def method_key(methods):
    # return sends the answer back to the caller. dict.get(name) retrieves a value and
    # returns None if the key is absent. Therefore a weight-only pathway has None for E
    # and ν. The generator inside tuple(...) repeats this lookup in the specified order.
    return tuple(methods.get(parameter) for parameter in
                 ["density", "elastic_modulus", "poissons_ratio"])

# These braces contain sets, which store unique items without a required order. The
# "expression for item in collection" syntax is a comprehension: it builds a collection
# by repeating an expression.
# The three nested for clauses generate all 4 × 4 × 2 method combinations. None means
# "no value" rather than a numerical zero.
expected_keys = {
    WEIGHT: {(density, None, None) for density in density_methods},
    ELASTICITY: {(density, elastic, poisson) for density in density_methods
                for elastic in elastic_methods for poisson in poisson_methods},
}
# This for loop checks each target in turn. Writing target, count unpacks each two-item
# tuple into two variable names.
for target, count in [(WEIGHT, 4), (ELASTICITY, 32)]:
    # Each path is a dictionary. path["methods"] retrieves its method choices;
    # method_key(...) gives them a consistent order for comparison.
    actual_keys = [method_key(path["methods"]) for path in inventory[target]]
    # assert stops the notebook with an AssertionError if its condition is false.
    # len(...) counts items; set(...) removes duplicates. The chained equality requires
    # the expected count AND the absence of duplicate method combinations.
    assert len(actual_keys) == count == len(set(actual_keys))
    # Equal counts alone could hide a missing pathway replaced by an incorrect one. Set
    # equality also checks that the actual method combinations are exactly the expected
    # combinations.
    assert set(actual_keys) == expected_keys[target]

# Make a display table from a list of row dictionaries. The two for clauses visit each
# target and then each pathway for that target. .items() supplies dictionary keys and
# values together.
# The expression "A if condition else B" chooses one value. Here it labels the single
# incompatible grain-domain combination. get(..., "—") supplies a readable dash when a
# pathway has no E or ν method.
inventory_table = pd.DataFrame([
    {"target": target, "density": path["methods"]["density"],
     "E method": path["methods"].get("elastic_modulus", "—"),
     "ν method": path["methods"].get("poissons_ratio", "—"),
     "feasibility": "Incompatible grain domains" if method_key(path["methods"]) ==
         ("kim_jamieson_table6", "bergfeld", "kochle") else "Potentially feasible"}
    for target, paths in inventory.items() for path in paths
])
# The .style methods change the displayed appearance only. hide(axis="index") hides
# Pandas' extra row numbers; the pathway data are unchanged.
display(inventory_table.style.hide(axis="index"))
# print writes a plain-text status message below the cell. Reaching this line means the
# preceding inventory assertions passed.
print("Enumerated 4 weight pathways and 32 weight-with-elasticity pathways, without duplicates.")

target,density,E method,ν method,feasibility
slab_weight_shear,data_flow,—,—,Potentially feasible
slab_weight_shear,geldsetzer,—,—,Potentially feasible
slab_weight_shear,kim_jamieson_table2,—,—,Potentially feasible
slab_weight_shear,kim_jamieson_table6,—,—,Potentially feasible
slab_weight_shear_with_elasticity,data_flow,bergfeld,kochle,Potentially feasible
slab_weight_shear_with_elasticity,data_flow,bergfeld,srivastava,Potentially feasible
slab_weight_shear_with_elasticity,geldsetzer,bergfeld,srivastava,Potentially feasible
slab_weight_shear_with_elasticity,kim_jamieson_table2,bergfeld,srivastava,Potentially feasible
slab_weight_shear_with_elasticity,kim_jamieson_table6,bergfeld,srivastava,Potentially feasible
slab_weight_shear_with_elasticity,geldsetzer,bergfeld,kochle,Potentially feasible


Enumerated 4 weight pathways and 32 weight-with-elasticity pathways, without duplicates.


### Comparison helper

The helper below checks supplied answers without calculating any manual mechanical
parameters. Comparisons use `rtol=1e-10`, `atol=1e-12` on both nominal values and
standard deviations. A missing output passes only when the independent manual
calculation and the framework both predict it is unavailable.

Each selected dataset slab is copied before framework execution. Layer weight
contributions are reconstructed from framework outputs because the graph stores
their sum rather than individual contributions. Every actual graph output is
also compared with its computation trace.

In [332]:
# This function produces one table row from a manual value, manual standard deviation,
# and framework output. It compares supplied answers but implements no manual physical
# equations.
# The optional argument expected_unavailable=False normally requires numbers. Set it
# to True when the independent manual calculation predicts no output. The line below is a
# docstring: a short built-in description of the function.
def comparison_row(parameter, layer, unit, manual_value, manual_sd,
                   framework_value, expected_unavailable=False):
    """Compare supplied answers; no manual equations are implemented here."""
    # An expected failure passes only if BOTH sides have no value. "is None" checks for
    # missing output; missing values are not treated as zero or accepted as numerical
    # agreement.
    if expected_unavailable:
        passed = manual_value is None and manual_sd is None and framework_value is None
        # Assign np.nan ("not a number") to the numeric display columns for unavailable
        # outputs. These placeholders will appear as dashes, and their status explicitly
        # records the expected failure.
        framework_nominal = framework_sd = delta_value = delta_sd = np.nan
        # Choose the status text using an if/else expression. An unexpected available
        # output is a failure of this verification.
        status = "EXPECTED FAILURE" if passed else "FAIL"
    # For every ordinary comparison, require real outputs from both calculations before
    # checking agreement.
    else:
        # The expression after the comma is an error message if the assertion fails. It
        # identifies the parameter so a failed notebook run can be diagnosed.
        assert manual_value is not None and manual_sd is not None, parameter
        assert framework_value is not None, (parameter, layer)
        # An uncertain number exposes .nominal_value and .std_dev. getattr(object, name,
        # default) uses the default if that attribute is absent, so ordinary floats are
        # also accepted. float(...) converts the extracted value to an ordinary Python
        # number.
        framework_nominal = float(getattr(framework_value, "nominal_value", framework_value))
        # An ordinary float has no stored uncertainty; its standard deviation is taken
        # as zero. An uncertain framework output supplies its actual propagated standard
        # deviation.
        framework_sd = float(getattr(framework_value, "std_dev", 0.0))
        # np.isfinite checks that each number is neither NaN nor infinity. .all()
        # requires every check to be True; invalid outputs cannot pass the later
        # tolerance test.
        assert np.isfinite([manual_value, manual_sd, framework_nominal, framework_sd]).all()
        # abs(...) gives a nonnegative absolute difference. Compare the nominal values
        # and standard deviations separately.
        delta_value = abs(float(manual_value) - framework_nominal)
        delta_sd = abs(float(manual_sd) - framework_sd)
        # np.isclose(a, b) checks |a − b| <= ATOL + RTOL × |b|. Both the value and its
        # standard deviation must satisfy this tolerance. "and" combines the two checks;
        # bool(...) stores an ordinary True/False result.
        passed = bool(
            np.isclose(manual_value, framework_nominal, rtol=RTOL, atol=ATOL)
            and np.isclose(manual_sd, framework_sd, rtol=RTOL, atol=ATOL)
        )
        # Turn the Boolean comparison result into the readable PASS or FAIL label.
        status = "PASS" if passed else "FAIL"
    # Return a dictionary that becomes one DataFrame row. Python indexes layers from
    # zero; adding 1 gives reader-facing labels 1, 2, and so on. layer=None identifies a
    # whole-slab quantity.
    return {
        "parameter": parameter, "layer": "slab" if layer is None else layer + 1,
        "unit": unit, "manual value": manual_value, "framework value": framework_nominal,
        "manual σ": manual_sd, "framework σ": framework_sd,
        "|Δ value|": delta_value, "|Δ σ|": delta_sd, "status": status, "passed": passed,
    }

## 2. Dataset and verification procedure

This analysis uses the **same ECTP slab cohort as the paper analysis**: all CAAML
pits in `examples/data`, with one slab above the failure layer for each propagating
Extended Column Test. Multiple tests in one pit can produce distinct slabs with
shared observations; these examples verify calculations, not independent statistical
replicates. Pit ID, slab ID, and test index identify every selected example.

For **each of the four weight and 32 elasticity pathways**, we first execute the
framework on the entire cohort and save the indices of **every successful slab**.
We then draw one successful slab, when that set is nonempty, and independently draw
ten slabs from the complete cohort, without a success filter. The ten draws are
without replacement within each pathway; they may overlap other pathways or the
successful example. A fixed seed makes all selections reproducible.

Each selected slab is recalculated by hand from its original observations and run
through `ExecutionEngine.execute_single` on a fresh copy. We compare every layer's
density, applicable hardness conversion, weight contribution, and (when requested)
$E$ and $\nu$, plus integrated weight, shear weight, and the final target.
The weight contribution has no separate graph trace; its framework-side value is
reconstructed explicitly from the framework's density and observed thickness.
Every graph output is also checked against its computation trace.

Use **measurement uncertainty only**, exact empirical coefficients, independent layer
observations, and one shared slope angle. We read the dataset's stored measurement
standard deviations (5% thickness, 10% measured density, 0.5 mm grain size, and
2° slope where observations are present), and independently assign hardness-index
uncertainty 0.67. Missing or nonfinite observations are not replaced or imputed.

A failed slab still contributes all available upstream calculations. A comparison
marked **EXPECTED FAILURE** means both the independent hand calculation and the
framework produce no value; the manual reason and the actual framework error are
shown separately. A numeric disagreement or an unexpected failure stops execution.
Successful samples are expanded below; the ten unfiltered examples appear in
expandable panels to keep the notebook readable. Their complete outputs are saved.

In [333]:
# Load the paper's observations. These utilities only parse data and define slabs;
# they are NOT used to calculate any manual mechanical parameter.
from copy import deepcopy
import random
import html
from IPython.display import HTML

# The examples folder contains the same loading helpers used in loss-analysis.ipynb.
sys.path.insert(0, str(repo_root / "examples"))
from notebook_utils import load_pits, create_ectp_slabs

dataset_pits = load_pits(str(repo_root / "examples" / "data"))
dataset_slabs = create_ectp_slabs(dataset_pits)
# Check the paper cohort and unique IDs. An updated dataset requires explicitly
# revisiting these counts rather than silently changing the verification population.
assert len(dataset_pits) == 50_278
assert len(dataset_slabs) == 14_776
assert len({slab.slab_id for slab in dataset_slabs}) == len(dataset_slabs)
DATASET_SEED = 20261010
print(f"Loaded {len(dataset_pits):,} pits and {len(dataset_slabs):,} ECTP slabs.")
print(f"Reproducible random seed: {DATASET_SEED}")

def dataset_snapshot(value):
    """Record source data, giving missing NaN values a stable comparison label."""
    # NaN does not equal itself. Use text for it, so before/after comparisons can
    # still detect changes in slabs with an unreported slope or other measurement.
    if isinstance(value, UFloat):
        return (dataset_snapshot(value.nominal_value), dataset_snapshot(value.std_dev))
    if isinstance(value, (float, np.floating)) and math.isnan(value):
        return "<NaN>"
    if is_dataclass(value):
        return tuple((field.name, dataset_snapshot(getattr(value, field.name)))
                     for field in fields(value))
    if isinstance(value, (list, tuple)):
        return tuple(dataset_snapshot(item) for item in value)
    if isinstance(value, dict):
        return tuple((key, dataset_snapshot(item)) for key, item in sorted(value.items()))
    return value

# Keep a complete before-state, including calculated fields and slab metadata.
# This also detects mutations of layers shared by several slabs from the same pit.
dataset_before = [dataset_snapshot(slab) for slab in dataset_slabs]

# A pathway is identified by target AND its density/E/ν method tuple. The same
# density choice appears in both targets, so target must be part of the key.
dataset_paths = []
for dataset_target in [WEIGHT, ELASTICITY]:
    for dataset_path in inventory[dataset_target]:
        dataset_paths.append((dataset_target, dataset_path))
dataset_successful_indices = {
    (target_name, method_key(path_info["methods"])): []
    for target_name, path_info in dataset_paths
}

# Scan EVERY slab through EVERY enumerated pathway. execute_all reuses shared
# density calculations within one slab to reduce work. The selected examples below
# use execute_single instead, so each detailed comparison starts with a clear cache.
for slab_index, observed_slab in enumerate(dataset_slabs):
    for target_name in [WEIGHT, ELASTICITY]:
        cohort_results = engine.execute_all(observed_slab, target_name, config=config)
        assert len(cohort_results.pathways) == len(inventory[target_name])
        seen_keys = set()
        for cohort_result in cohort_results.pathways.values():
            key = (target_name, method_key(cohort_result.methods_used))
            assert key in dataset_successful_indices and key not in seen_keys
            seen_keys.add(key)
            if cohort_result.success:
                # A successful final target must have finite values/uncertainties
                # throughout its trace, not just a truthy success flag.
                for trace in cohort_result.computation_trace:
                    assert trace.success and trace.output is not None
                    assert np.isfinite([
                        float(getattr(trace.output, "nominal_value", trace.output)),
                        float(getattr(trace.output, "std_dev", 0.0)),
                    ]).all()
                dataset_successful_indices[key].append(slab_index)
    if (slab_index + 1) % 3000 == 0:
        print(f"Scanned {slab_index + 1:,} / {len(dataset_slabs):,} slabs.", flush=True)

# Indices specify the complete set, not just the displayed examples. These lists
# let a reader retrieve any member via dataset_slabs[index] without rerunning the scan.
dataset_successful_ids = {}
dataset_cohort_rows = []
for target_name, path_info in dataset_paths:
    key = (target_name, method_key(path_info["methods"]))
    indices = dataset_successful_indices[key]
    assert len(indices) == len(set(indices))
    dataset_successful_ids[key] = [dataset_slabs[index].slab_id for index in indices]
    dataset_cohort_rows.append({
        "target": target_name, "density": key[1][0],
        "E method": key[1][1] or "—", "ν method": key[1][2] or "—",
        "successful slabs": len(indices), "total slabs": len(dataset_slabs),
    })
dataset_cohort_table = pd.DataFrame(dataset_cohort_rows)
display(dataset_cohort_table.style.hide(axis="index"))
assert [dataset_snapshot(slab) for slab in dataset_slabs] == dataset_before
print("Complete successful sets recorded for all 36 pathways; source slabs unchanged.")

Loaded 50,278 pits and 14,776 ECTP slabs.
Reproducible random seed: 20261010
Scanned 3,000 / 14,776 slabs.
Scanned 6,000 / 14,776 slabs.
Scanned 9,000 / 14,776 slabs.
Scanned 12,000 / 14,776 slabs.


target,density,E method,ν method,successful slabs,total slabs
slab_weight_shear,data_flow,—,—,109,14776
slab_weight_shear,geldsetzer,—,—,4187,14776
slab_weight_shear,kim_jamieson_table2,—,—,5470,14776
slab_weight_shear,kim_jamieson_table6,—,—,697,14776
slab_weight_shear_with_elasticity,data_flow,bergfeld,kochle,6,14776
slab_weight_shear_with_elasticity,data_flow,bergfeld,srivastava,8,14776
slab_weight_shear_with_elasticity,geldsetzer,bergfeld,srivastava,349,14776
slab_weight_shear_with_elasticity,kim_jamieson_table2,bergfeld,srivastava,380,14776
slab_weight_shear_with_elasticity,kim_jamieson_table6,bergfeld,srivastava,0,14776
slab_weight_shear_with_elasticity,geldsetzer,bergfeld,kochle,443,14776


Complete successful sets recorded for all 36 pathways; source slabs unchanged.


## 3. Independent equations for arbitrary dataset layers

Dataset slabs can contain any number of layers and several grain classes. The following coefficient tables and functions **belong to
this notebook**. They use ordinary Python numbers and explicitly coded analytical
derivatives; they do not import framework equations, constants, grain resolvers,
or uncertainty propagation. Reading an observation's nominal value and stored
standard deviation is the only use of its uncertain-number attributes on the
manual side. Density subtypes are matched first; an unlisted subtype falls back to
its first two letters only when that basic class is supported. Elastic methods use
the basic grain class. Validity restrictions apply to nominal inputs, as above.

The displayed coefficient tables include all supported classes and hardness ranges.
Density follows one of

$$
\rho=\rho_{\rm measured},\quad A+BH,\quad
154+1.51H^{3.15},\quad91.8e^{0.270H},\quad AH+B d+C.
$$

Their measurement standard deviations are, respectively,
$\sigma_{\rho,\mathrm{measured}}$, $|B|\sigma_H$,
$|1.51(3.15)H^{2.15}|\sigma_H$,
$|0.270\rho|\sigma_H$, and
$\sqrt{(A\sigma_H)^2+(B\sigma_d)^2}$.
For a power-law modulus $E=C(\rho/917)^n$,
$\sigma_E=|Cn(\rho/917)^{n-1}/917|\sigma_\rho$;
for $E=a e^{b\rho}$, $\sigma_E=|bE|\sigma_\rho$.
The selected Poisson constants have $\sigma_\nu=0$ when empirical uncertainty is
disabled. No uncertainty is assigned to categorical grain forms or domain selection.

With $t_i=h_i/100$, $\sigma_{t_i}=\sigma_{h_i}/100$, and $g=9.81$,

$$
w_i=9.81\rho_i t_i,\quad
\sigma_{w_i}^2=(9.81t_i\sigma_{\rho_i})^2+
(9.81\rho_i\sigma_{t_i})^2,\quad
W=\sum_i w_i,\quad \sigma_W^2=\sum_i\sigma_{w_i}^2.
$$

$$
\theta_r=\theta\pi/180,\quad\sigma_{\theta_r}=\sigma_\theta\pi/180,
\quad W_s=W\sin\theta_r,\quad
\sigma_{W_s}^2=(\sin\theta_r\,\sigma_W)^2+
(W\cos\theta_r\,\sigma_{\theta_r})^2.
$$

The elasticity target equals $W_s$ only when every layer has valid $E$ and
$\nu$. This target does not compute shear modulus or laminate stiffnesses.
Each selected slab's substitution table shows the actual numbers used in these
equations, including the uncertainty equation or the reason a step is unavailable.

In [334]:
# Define every hardness conversion locally. None (missing) is distinct from zero.
dataset_hardness = {
    "F-": 0.67, "F": 1.00, "F+": 1.33,
    "4F-": 1.67, "4F": 2.00, "4F+": 2.33,
    "1F-": 2.67, "1F": 3.00, "1F+": 3.33,
    "P-": 3.67, "P": 4.00, "P+": 4.33,
    "K-": 4.67, "K": 5.00, "K+": 5.33,
}

# Each density entry is (equation type, coefficients, lowest H, highest H).
# "linear" means A + B*H; "power" means A + B*H**n;
# "exp" means A*exp(B*H); "size" means A*H + B*grain_size + C.
# Coefficients are ordinary floats and are treated as exact.
dataset_density_coefficients = {
    "geldsetzer": {
        "PP": ("linear", (45.0, 36.0), 0.67, 4.00),
        "PPgp": ("linear", (83.0, 37.0), 0.67, 4.00),
        "DF": ("linear", (65.0, 36.0), 0.67, 4.33),
        "RG": ("power", (154.0, 1.51, 3.15), 1.00, 5.33),
        "FC": ("linear", (112.0, 46.0), 0.67, 4.67),
        "DH": ("linear", (185.0, 25.0), 1.00, 5.00),
    },
    "kim_jamieson_table2": {
        "PP": ("linear", (41.3, 40.3), 0.67, 4.00),
        "PPgp": ("linear", (61.8, 46.4), 0.67, 4.00),
        "DF": ("linear", (62.5, 37.4), 0.67, 4.67),
        "RG": ("exp", (91.8, 0.270), 0.67, 5.33),
        "RGxf": ("linear", (85.0, 46.3), 0.67, 4.33),
        "FC": ("linear", (103.0, 50.6), 0.67, 5.00),
        "FCxr": ("linear", (68.8, 58.6), 0.67, 5.33),
        "DH": ("linear", (214.0, 19.0), 1.00, 5.00),
        "MFcr": ("linear", (235.0, 15.1), 2.00, 5.33),
    },
    "kim_jamieson_table6": {
        "FC": ("size", (51.9, 19.7, 82.8), 1.67, 4.00),
        "FCxr": ("size", (60.4, 27.7, 36.7), 2.33, 4.33),
        "PP": ("size", (40.0, -7.33, 52.8), 0.67, 2.00),
        "PPgp": ("size", (38.8, 18.8, 35.7), 1.00, 3.33),
        "DF": ("size", (37.9, -8.87, 71.4), 1.00, 3.00),
        "MF": ("size", (34.9, 11.2, 124.5), 2.33, 4.33),
    },
}
# Display the coefficients rather than leaving grain-specific choices hidden in code.
display(pd.DataFrame([
    {"method": method, "grain": grain, "equation": item[0],
     "coefficients": str(item[1]), "min H": item[2], "max H": item[3]}
    for method, table in dataset_density_coefficients.items()
    for grain, item in table.items()
]).style.hide(axis="index"))

# E coefficients and domains. A power model uses (C, n), so C already includes
# the ice modulus where required. Köchle has two explicitly defined branches.
dataset_power_coefficients = {
    "bergfeld": {grain: (6500.0, 4.4) for grain in ["PP", "RG", "DF"]},
    "wautier": {grain: (10000.0 * 0.78, 2.34)
                for grain in ["DF", "RG", "FC", "DH", "MF"]},
    "schottner": {
        "DF": (10000.0 * 0.40, 4.6), "RG": (10000.0 * 0.40, 4.6),
        "FC": (10000.0 * 1.8, 5.1), "DH": (10000.0 * 1.8, 5.1),
        "SH": (10000.0 * 0.011, 1.7),
    },
}
dataset_power_domains = {"bergfeld": (110.0, 363.0), "wautier": (103.0, 544.0)}
display(pd.DataFrame([
    {"method": method, "grain": grain, "C (MPa)": C, "n": exponent,
     "density domain (kg/m³)": str(dataset_power_domains.get(method, "No fitted bound"))}
    for method, table in dataset_power_coefficients.items()
    for grain, (C, exponent) in table.items()
] + [
    {"method": "kochle", "grain": "RG, FC, DH, MF", "C (MPa)": np.nan,
     "n": np.nan, "density domain (kg/m³)":
     "150 ≤ ρ < 250: 0.0061 exp(0.0396ρ); 250 ≤ ρ ≤ 450: 6.0457 exp(0.011ρ)"}
]).style.hide(axis="index").format(na_rep="—"))

dataset_poisson_coefficients = {
    "kochle": {"RG": 0.171, "FC": 0.130, "DH": 0.087},
    "srivastava": {"RG": 0.191, "PP": 0.132, "DF": 0.132, "FC": 0.170, "DH": 0.170},
}
display(pd.DataFrame([
    {"method": method, "grain": grain, "ν": value, "σν": 0.0,
     "density domain (kg/m³)": "Not required" if method == "kochle" else
         ("200 < ρ ≤ 580" if grain == "RG" else "ρ > 200")}
    for method, table in dataset_poisson_coefficients.items()
    for grain, value in table.items()
]).style.hide(axis="index"))

method,grain,equation,coefficients,min H,max H
geldsetzer,PP,linear,"(45.0, 36.0)",0.670000,4.000000
geldsetzer,PPgp,linear,"(83.0, 37.0)",0.670000,4.000000
geldsetzer,DF,linear,"(65.0, 36.0)",0.670000,4.330000
geldsetzer,RG,power,"(154.0, 1.51, 3.15)",1.000000,5.330000
geldsetzer,FC,linear,"(112.0, 46.0)",0.670000,4.670000
geldsetzer,DH,linear,"(185.0, 25.0)",1.000000,5.000000
kim_jamieson_table2,PP,linear,"(41.3, 40.3)",0.670000,4.000000
kim_jamieson_table2,PPgp,linear,"(61.8, 46.4)",0.670000,4.000000
kim_jamieson_table2,DF,linear,"(62.5, 37.4)",0.670000,4.670000
kim_jamieson_table2,RG,exp,"(91.8, 0.27)",0.670000,5.330000


method,grain,C (MPa),n,density domain (kg/m³)
bergfeld,PP,6500.000000,4.400000,"(110.0, 363.0)"
bergfeld,RG,6500.000000,4.400000,"(110.0, 363.0)"
bergfeld,DF,6500.000000,4.400000,"(110.0, 363.0)"
wautier,DF,7800.000000,2.340000,"(103.0, 544.0)"
wautier,RG,7800.000000,2.340000,"(103.0, 544.0)"
wautier,FC,7800.000000,2.340000,"(103.0, 544.0)"
wautier,DH,7800.000000,2.340000,"(103.0, 544.0)"
wautier,MF,7800.000000,2.340000,"(103.0, 544.0)"
schottner,DF,4000.000000,4.600000,No fitted bound
schottner,RG,4000.000000,4.600000,No fitted bound


method,grain,ν,σν,density domain (kg/m³)
kochle,RG,0.171000,0.000000,Not required
kochle,FC,0.130000,0.000000,Not required
kochle,DH,0.087000,0.000000,Not required
srivastava,RG,0.191000,0.000000,200 < ρ ≤ 580
srivastava,PP,0.132000,0.000000,ρ > 200
srivastava,DF,0.132000,0.000000,ρ > 200
srivastava,FC,0.170000,0.000000,ρ > 200
srivastava,DH,0.170000,0.000000,ρ > 200


In [335]:
def dataset_answer(value=None, sd=None, substitution="", uncertainty="", reason=""):
    """A small dictionary stores one manual answer and its explanation."""
    # Only valid answers contain numbers. Unavailable steps explicitly contain None
    # for both nominal value and standard deviation, with an explanatory reason.
    return {"value": value, "sd": sd, "substitution": substitution,
            "uncertainty": uncertainty, "reason": reason}

def dataset_measurement(observation, name):
    """Extract an original measurement as two plain numbers; do no propagation."""
    if observation is None:
        return dataset_answer(reason=f"Missing {name}")
    value = float(getattr(observation, "nominal_value", observation))
    sd = float(getattr(observation, "std_dev", 0.0))
    if not math.isfinite(value) or not math.isfinite(sd):
        return dataset_answer(reason=f"Nonfinite {name}")
    return dataset_answer(value, sd, f"Observed {name} = {value:.9g}",
                          f"Stored measurement σ = {sd:.9g}")

def dataset_manual_hardness(layer):
    """Convert the reported hardness code using our independent lookup table."""
    value = dataset_hardness.get(layer.hand_hardness)
    if value is None:
        return dataset_answer(reason=f"Missing/unsupported hardness: {layer.hand_hardness!r}")
    return dataset_answer(value, 0.67, f"H({layer.hand_hardness}) = {value:g}", "σH = 0.67")

def dataset_manual_density(layer, method):
    """Calculate density and its analytical measurement uncertainty."""
    if method == "data_flow":
        return dataset_measurement(layer.density_measured, "measured density (kg/m³)")
    H = dataset_manual_hardness(layer)
    if H["value"] is None:
        return dataset_answer(reason=H["reason"])
    table = dataset_density_coefficients[method]
    grain = layer.grain_form
    # Try the full subtype first. If absent, only its basic two-letter class may
    # be used. Unsupported classes remain unavailable; we never invent a default.
    if grain not in table:
        grain = grain[:2] if grain else None
    if grain not in table:
        return dataset_answer(reason=f"{method} does not support grain {layer.grain_form!r}")
    equation, coefficients, minimum, maximum = table[grain]
    h, sh = H["value"], H["sd"]
    if not minimum <= h <= maximum:
        return dataset_answer(reason=f"H={h:g} outside [{minimum:g}, {maximum:g}] for {grain}")
    A, B = coefficients[:2]
    if equation == "linear":
        rho = A + B * h
        sd = abs(B) * sh
        substitution = f"ρ = {A:g} + {B:g} × {h:g} = {rho:.9g}"
        uncertainty = f"σρ = |{B:g}| × {sh:g} = {sd:.9g}"
    elif equation == "power":
        exponent = coefficients[2]
        rho = A + B * h ** exponent
        derivative = B * exponent * h ** (exponent - 1)
        sd = abs(derivative) * sh
        substitution = f"ρ = {A:g} + {B:g} × {h:g}^{exponent:g} = {rho:.9g}"
        uncertainty = f"σρ = |{B:g} × {exponent:g} × {h:g}^{exponent-1:g}| × {sh:g} = {sd:.9g}"
    elif equation == "exp":
        rho = A * math.exp(B * h)
        sd = abs(B * rho) * sh
        substitution = f"ρ = {A:g} exp({B:g} × {h:g}) = {rho:.9g}"
        uncertainty = f"σρ = |{B:g} × {rho:.9g}| × {sh:g} = {sd:.9g}"
    else:
        # The multivariable regression needs both independent H and grain size.
        size = dataset_measurement(layer.grain_size_avg, "grain size (mm)")
        if size["value"] is None:
            return dataset_answer(reason=size["reason"])
        C = coefficients[2]
        d, sd_size = size["value"], size["sd"]
        rho = A * h + B * d + C
        sd = math.sqrt((A * sh)**2 + (B * sd_size)**2)
        substitution = f"ρ = {A:g} × {h:g} + {B:g} × {d:g} + {C:g} = {rho:.9g}"
        uncertainty = f"σρ = sqrt(({A:g} × {sh:g})² + ({B:g} × {sd_size:g})²) = {sd:.9g}"
    return dataset_answer(rho, sd, f"{grain}: {substitution}", uncertainty)

def dataset_manual_modulus(layer, density, method):
    """Calculate E in MPa; use only the manually calculated density."""
    if density["value"] is None:
        return dataset_answer(reason="E requires an available pathway density")
    if not layer.grain_form:
        return dataset_answer(reason="E requires a grain form")
    grain = layer.grain_form[:2].upper()
    rho, srho = density["value"], density["sd"]
    if method == "kochle":
        if grain not in ["RG", "FC", "DH", "MF"]:
            return dataset_answer(reason=f"Köchle E does not support {grain}")
        if 150.0 <= rho < 250.0:
            A, B = 0.0061, 0.0396
        elif 250.0 <= rho <= 450.0:
            A, B = 6.0457, 0.011
        else:
            return dataset_answer(reason=f"ρ={rho:.9g} outside Köchle E range [150, 450]")
        E = A * math.exp(B * rho)
        sd = abs(B * E) * srho
        return dataset_answer(E, sd,
            f"E = {A:g} exp({B:g} × {rho:.9g}) = {E:.9g}",
            f"σE = |{B:g} × {E:.9g}| × {srho:.9g} = {sd:.9g}")
    table = dataset_power_coefficients[method]
    if grain not in table:
        return dataset_answer(reason=f"{method} E does not support {grain}")
    if method in dataset_power_domains:
        minimum, maximum = dataset_power_domains[method]
        if not minimum <= rho <= maximum:
            return dataset_answer(reason=f"ρ={rho:.9g} outside {method} range [{minimum:g}, {maximum:g}]")
    if rho < 0:
        return dataset_answer(reason="Negative density gives a non-real fractional power")
    C, exponent = table[grain]
    E = C * (rho / 917.0) ** exponent
    # Use the derivative in power form rather than n*E/rho, which divides by zero
    # if an unbounded model receives zero density.
    derivative = C * exponent * (rho / 917.0) ** (exponent - 1) / 917.0
    sd = abs(derivative) * srho
    return dataset_answer(E, sd,
        f"E = {C:g} × ({rho:.9g}/917)^{exponent:g} = {E:.9g}",
        f"σE = |{C:g} × {exponent:g} × ({rho:.9g}/917)^{exponent-1:g}/917| × {srho:.9g} = {sd:.9g}")

def dataset_manual_poisson(layer, density, method):
    """Check the applicable domain, then return the exact selected ν constant."""
    grain = layer.grain_form[:2].upper() if layer.grain_form else None
    table = dataset_poisson_coefficients[method]
    if grain not in table:
        return dataset_answer(reason=f"{method} ν does not support {grain!r}")
    # Köchle ν uses grain form only: a failed density need not prevent this step.
    if method == "srivastava":
        if density["value"] is None:
            return dataset_answer(reason="Srivastava ν requires an available pathway density")
        rho = density["value"]
        if rho <= 200.0 or (grain == "RG" and rho > 580.0):
            return dataset_answer(reason=f"ρ={rho:.9g} outside Srivastava ν domain for {grain}")
    return dataset_answer(table[grain], 0.0, f"ν({grain}) = {table[grain]:g}", "σν = 0 (exact empirical constant)")

In [336]:
def dataset_manual_pathway(observed_slab, target_name, methods):
    """Calculate every step from observations, including available upstream steps."""
    # Dictionaries keyed by (parameter, layer index) mirror the meaning of graph
    # steps without using the graph to calculate manual values. None denotes a slab.
    answers = {}
    contributions = []
    for layer_index, layer in enumerate(observed_slab.layers):
        if methods["density"] != "data_flow":
            answers[("hand_hardness_index", layer_index)] = dataset_manual_hardness(layer)
        density = dataset_manual_density(layer, methods["density"])
        answers[("density", layer_index)] = density
        thickness = dataset_measurement(layer.thickness, "thickness (cm)")
        if density["value"] is None or thickness["value"] is None:
            contribution = dataset_answer(reason="Layer weight requires valid density and thickness")
        else:
            rho, srho = density["value"], density["sd"]
            # Convert both thickness and its standard deviation from cm to metres.
            t, st = thickness["value"] / 100.0, thickness["sd"] / 100.0
            w = rho * t * 9.81
            sw = math.sqrt((9.81 * t * srho)**2 + (9.81 * rho * st)**2)
            contribution = dataset_answer(w, sw,
                f"w = {rho:.9g} × ({thickness['value']:.9g}/100) × 9.81 = {w:.9g}",
                f"σw = sqrt((9.81 × {t:.9g} × {srho:.9g})² + (9.81 × {rho:.9g} × {st:.9g})²) = {sw:.9g}")
        contributions.append(contribution)
        answers[("layer_weight", layer_index)] = contribution
        if target_name == ELASTICITY:
            answers[("elastic_modulus", layer_index)] = dataset_manual_modulus(
                layer, density, methods["elastic_modulus"])
            answers[("poissons_ratio", layer_index)] = dataset_manual_poisson(
                layer, density, methods["poissons_ratio"])

    if contributions and all(item["value"] is not None for item in contributions):
        W = sum(item["value"] for item in contributions)
        sW = math.sqrt(sum(item["sd"]**2 for item in contributions))
        weight = dataset_answer(W, sW,
            "W = " + " + ".join(f"{item['value']:.9g}" for item in contributions) + f" = {W:.9g}",
            "σW = sqrt(" + " + ".join(f"{item['sd']:.9g}²" for item in contributions) + f") = {sW:.9g}")
    else:
        weight = dataset_answer(reason="W requires a valid weight contribution on every slab layer")
    answers[("slab_weight", None)] = weight
    angle = dataset_measurement(observed_slab.angle, "slope angle (degrees)")
    if weight["value"] is None or angle["value"] is None:
        shear = dataset_answer(reason="Shear weight requires valid W and slope angle")
    else:
        theta = angle["value"] * math.pi / 180.0
        stheta = angle["sd"] * math.pi / 180.0
        W, sW = weight["value"], weight["sd"]
        Ws = W * math.sin(theta)
        sWs = math.sqrt((math.sin(theta) * sW)**2 + (W * math.cos(theta) * stheta)**2)
        shear = dataset_answer(Ws, sWs,
            f"Ws = {W:.9g} sin({angle['value']:.9g} × π/180) = {Ws:.9g}",
            f"σWs = sqrt((sin({theta:.9g}) × {sW:.9g})² + ({W:.9g} × cos({theta:.9g}) × {stheta:.9g})²) = {sWs:.9g}")
    answers[(WEIGHT, None)] = shear
    if target_name == ELASTICITY:
        complete = all(answers[(parameter, index)]["value"] is not None
                       for parameter in ["elastic_modulus", "poissons_ratio"]
                       for index in range(len(observed_slab.layers)))
        if complete and shear["value"] is not None:
            answers[(ELASTICITY, None)] = dataset_answer(
                shear["value"], shear["sd"], "All layers have E and ν: target = Ws",
                "Target σ = σWs (same quantity)")
        else:
            answers[(ELASTICITY, None)] = dataset_answer(
                reason="Elasticity target requires valid Ws, E and ν on every layer")
    return answers

def dataset_framework_value(result, parameter, layer_index):
    """Read framework results; reconstruct only the non-traced layer contribution."""
    if layer_index is None:
        return getattr(result.slab, parameter)
    layer = result.slab.layers[layer_index]
    if parameter == "density":
        return layer.density_calculated
    if parameter == "layer_weight":
        if layer.density_calculated is None or layer.thickness is None:
            return None
        # This product intentionally uses framework uncertain arithmetic. The manual
        # counterpart above uses only plain numbers and an explicit variance equation.
        value = layer.density_calculated * (layer.thickness / 100.0) * 9.81
        if not np.isfinite([float(getattr(value, "nominal_value", value)),
                              float(getattr(value, "std_dev", 0.0))]).all():
            return None
        return value
    return getattr(layer, parameter)

dataset_units = {
    "hand_hardness_index": "1", "density": "kg/m³", "layer_weight": "Pa",
    "elastic_modulus": "MPa", "poissons_ratio": "1", "slab_weight": "Pa",
    WEIGHT: "Pa", ELASTICITY: "Pa",
}

def dataset_verify_slab(slab_index, target_name, path_info, selection):
    """Run one independent manual/framework comparison and require exact coverage."""
    original = dataset_slabs[slab_index]
    before = dataset_snapshot(original)
    methods = path_info["methods"]
    manual = dataset_manual_pathway(original, target_name, methods)
    # deepcopy creates separate nested Layer objects as well as a separate Slab.
    # Neither shared source layers nor any manual output becomes a framework input.
    fresh_input = deepcopy(original)
    fresh_before = dataset_snapshot(fresh_input)
    result = engine.execute_single(fresh_input, target_name, methods, config=config)
    assert result is not None
    assert result.pathway_id == path_info["id"] and result.methods_used == methods
    assert dataset_snapshot(original) == before
    assert dataset_snapshot(fresh_input) == fresh_before
    traces = {(trace.parameter, trace.layer_index): trace for trace in result.computation_trace}
    assert len(traces) == len(result.computation_trace), "Duplicate trace keys"
    # Hardness conversion and individual layer contributions are not graph nodes in
    # these pathways. All other manual entries must have exactly one graph trace.
    expected_trace_keys = {key for key in manual
                           if key[0] not in ["hand_hardness_index", "layer_weight"]}
    assert set(traces) == expected_trace_keys
    rows = []
    for (parameter, layer_index), answer in manual.items():
        framework_value = dataset_framework_value(result, parameter, layer_index)
        unavailable = answer["value"] is None
        row = comparison_row(parameter, layer_index, dataset_units[parameter],
                             answer["value"], answer["sd"], framework_value,
                             expected_unavailable=unavailable)
        trace = traces.get((parameter, layer_index))
        if trace is not None:
            assert trace.method_name == methods[parameter]
            assert trace.success == (not unavailable), (original.slab_id, parameter, trace.error)
            if unavailable:
                assert trace.output is None and trace.error
            else:
                trace_row = comparison_row(parameter, layer_index, dataset_units[parameter],
                                           answer["value"], answer["sd"], trace.output)
                assert trace_row["passed"], (original.slab_id, "trace", parameter)
        row["slab ID"] = original.slab_id
        row["selection"] = selection
        rows.append(row)
    comparison = pd.DataFrame(rows)
    assert comparison["passed"].all(), (original.slab_id, comparison.loc[~comparison["passed"]])
    manual_success = manual[(target_name, None)]["value"] is not None
    assert result.success == manual_success
    # Membership in the full cohort success set must agree with this fresh execution.
    cohort_key = (target_name, method_key(methods))
    assert (slab_index in dataset_successful_indices[cohort_key]) == manual_success
    if target_name == ELASTICITY and manual_success:
        assert manual[(ELASTICITY, None)]["value"] == manual[(WEIGHT, None)]["value"]
        assert manual[(ELASTICITY, None)]["sd"] == manual[(WEIGHT, None)]["sd"]
        # Compare with a separately executed weight-only pathway, including σ.
        weight_result = engine.execute_single(deepcopy(original), WEIGHT,
                                              {"density": methods["density"]}, config=config)
        assert weight_result is not None and weight_result.success
        same_weight = comparison_row("elasticity target = separate shear weight", None, "Pa",
            manual[(ELASTICITY, None)]["value"], manual[(ELASTICITY, None)]["sd"],
            weight_result.slab.slab_weight_shear)
        assert same_weight["passed"]
    return {"index": slab_index, "selection": selection, "slab": original,
            "manual": manual, "result": result, "comparison": comparison}

In [337]:
def dataset_html_table(table, numeric=False):
    """Format a compact, horizontally scrollable table without changing its data."""
    # HTML is text describing the display. escape=True prevents an observation or
    # error string from being interpreted as HTML. Rounding is for display only.
    rendered = table.to_html(index=False, escape=True, na_rep="—",
                             float_format=lambda value: f"{value:.9g}", border=0)
    return '<div style="overflow-x:auto; margin:12px 0">' + rendered + '</div>'

def dataset_render_slab(case, expanded=False):
    """Show observations, manual substitutions, comparisons, and failed traces."""
    slab = case["slab"]
    angle = dataset_measurement(slab.angle, "angle")
    observation_rows = []
    for index, layer in enumerate(slab.layers):
        row = {"layer": index + 1, "grain": layer.grain_form, "hardness": layer.hand_hardness}
        for field_name, label in [("thickness", "h (cm)"), ("grain_size_avg", "d (mm)"),
                                  ("density_measured", "measured ρ (kg/m³)")]:
            measurement = dataset_measurement(getattr(layer, field_name), label)
            row[label] = measurement["value"]
            row["σ " + label] = measurement["sd"]
        observation_rows.append(row)
    substitution_rows = []
    for (parameter, index), answer in case["manual"].items():
        substitution_rows.append({
            "parameter": parameter, "layer": "slab" if index is None else index + 1,
            "nominal substitution / reason": answer["substitution"] or answer["reason"],
            "standard-deviation substitution": answer["uncertainty"] or "Unavailable",
        })
    failures = [{"parameter": trace.parameter,
                 "layer": "slab" if trace.layer_index is None else trace.layer_index + 1,
                 "method": trace.method_name, "framework error": trace.error}
                for trace in case["result"].computation_trace if not trace.success]
    label = (f"{case['selection']}: {slab.slab_id} — {len(slab.layers)} layers; "
             f"target {'SUCCESS' if case['result'].success else 'UNAVAILABLE'}")
    # <details> gives the reader an expandable panel. The successful witness is open;
    # each unfiltered sample can be opened individually to inspect its saved tables.
    opening = " open" if expanded else ""
    parts = [f'<details{opening}><summary><strong>{html.escape(label)}</strong></summary>',
             f'<p>Pit: {html.escape(str(slab.pit_id))}; test index: {slab.test_result_index}; '
             f'dataset index: {case["index"]}; slope (degrees): '
             f'{angle["value"]} ± {angle["sd"]}.</p>',
             '<p><strong>Original measurements</strong></p>',
             dataset_html_table(pd.DataFrame(observation_rows)),
             '<p><strong>Independent hand calculations and analytical uncertainties</strong></p>',
             dataset_html_table(pd.DataFrame(substitution_rows)),
             '<p><strong>Manual / framework comparisons</strong></p>',
             dataset_html_table(case["comparison"].drop(columns=["passed", "slab ID", "selection"]))]
    if failures:
        parts.extend(['<p><strong>Framework failure traces (upstream successes are compared above)</strong></p>',
                      dataset_html_table(pd.DataFrame(failures))])
    else:
        parts.append('<p>All computation traces succeeded and their outputs agree with the hand calculations.</p>')
    parts.append('</details>')
    display(HTML("\n".join(parts)))

# Store all selected cases and comparisons for a final duplicate/coverage audit.
dataset_case_records = []
dataset_summary_records = []
dataset_selection_records = []

def dataset_verify_pathway(target_name, selected_methods, pathway_number):
    """Select and verify one successful witness plus ten unfiltered dataset slabs."""
    matching = [path_info for path_info in inventory[target_name]
                if method_key(path_info["methods"]) == method_key(selected_methods)]
    assert len(matching) == 1
    path_info = matching[0]
    key = (target_name, method_key(selected_methods))
    successful = dataset_successful_indices[key]
    # Use a fresh generator for each pathway. Adding its fixed ordinal to the base
    # seed ensures rerunning one section gives the same selections as Run All.
    rng = random.Random(DATASET_SEED + pathway_number)
    display(Markdown(f"**Successful set: {len(successful):,} / {len(dataset_slabs):,} slabs.** "
                     f"Complete indices and IDs are stored under `{key}` in "
                     "`dataset_successful_indices` and `dataset_successful_ids`."))
    selected = []
    if successful:
        selected.append((rng.choice(successful), "successful witness"))
    else:
        display(Markdown("**No successful slab exists in this dataset for this pathway.** "
                         "The ten unfiltered examples below still verify available upstream steps and failures."))
    # sample(..., 10) draws ten DISTINCT indices from the whole dataset. It does not
    # require success and does not remove the separately selected successful witness.
    unfiltered = rng.sample(range(len(dataset_slabs)), 10)
    assert len(unfiltered) == len(set(unfiltered)) == 10
    selected.extend((index, f"unfiltered {number}") for number, index in enumerate(unfiltered, 1))
    selection_rows = []
    for index, label in selected:
        slab = dataset_slabs[index]
        selection_rows.append({"selection": label, "dataset index": index,
            "pit ID": slab.pit_id, "slab ID": slab.slab_id, "layers": len(slab.layers),
            "target successful in full scan": index in successful})
        dataset_selection_records.append({"target": target_name, "methods": key[1],
                                         "pathway number": pathway_number,
                                         "selection": label, "dataset index": index})
    display(pd.DataFrame(selection_rows).style.hide(axis="index"))
    cases = []
    for index, label in selected:
        case = dataset_verify_slab(index, target_name, path_info, label)
        case["pathway number"] = pathway_number
        case["target"] = target_name
        case["methods"] = key[1]
        cases.append(case)
        dataset_case_records.append(case)
        dataset_render_slab(case, expanded=(label == "successful witness"))
    tables = pd.concat([case["comparison"] for case in cases], ignore_index=True)
    dataset_summary_records.append({
        "pathway number": pathway_number, "target": target_name, "density": key[1][0],
        "E method": key[1][1] or "—", "ν method": key[1][2] or "—",
        "successful set": len(successful), "successful witness": int(bool(successful)),
        "unfiltered draws": len(unfiltered),
        "unfiltered successes": sum(case["result"].success for case in cases
                                    if case["selection"] != "successful witness"),
        "numeric comparisons": int((tables["status"] == "PASS").sum()),
        "matched unavailable": int((tables["status"] == "EXPECTED FAILURE").sum()),
        "max |Δ value|": tables["|Δ value|"].max(), "max |Δ σ|": tables["|Δ σ|"].max(),
        "verification": "PASS", "pathway ID": path_info["id"],
    })
    print(f"Verified {len(cases)} selected slabs; {len(tables)} parameter comparisons; "
          "all values, standard deviations, trace outputs and availability checks agree.")

## 4. Weight pathways — actual slabs

### 4.1. Measured density → slab weight → shear weight

The successful witness and ten unfiltered draws use this density method. Expand a selected slab to inspect its original measurements, numerical substitutions, uncertainties, comparisons, and any failure traces.

In [338]:
# Select this weight pathway explicitly. The section can be rerun independently
# after the setup and helper cells; its seed is fixed by pathway number 1.
dataset_verify_pathway(
    target_name=WEIGHT,
    selected_methods={"density": "data_flow"},
    pathway_number=1,
)

**Successful set: 109 / 14,776 slabs.** Complete indices and IDs are stored under `('slab_weight_shear', ('data_flow', None, None))` in `dataset_successful_indices` and `dataset_successful_ids`.

selection,dataset index,pit ID,slab ID,layers,target successful in full scan
successful witness,6865,45273,45273_slab_0,2,True
unfiltered 1,5678,37702,37702_slab_2,8,False
unfiltered 2,8122,49586,49586_slab_0,3,False
unfiltered 3,5537,37030,37030_slab_1,4,False
unfiltered 4,4570,31841,31841_slab_0,4,False
unfiltered 5,898,14043,14043_slab_0,1,False
unfiltered 6,655,12901,12901_slab_0,3,False
unfiltered 7,5867,39036,39036_slab_0,6,False
unfiltered 8,7863,48575,48575_slab_0,5,False
unfiltered 9,609,1269,1269_slab_0,4,False


Verified 11 selected slabs; 118 parameter comparisons; all values, standard deviations, trace outputs and availability checks agree.


### 4.2. Geldsetzer → slab weight → shear weight

The successful witness and ten unfiltered draws use this density method. Expand a selected slab to inspect its original measurements, numerical substitutions, uncertainties, comparisons, and any failure traces.

In [339]:
# Select this weight pathway explicitly. The section can be rerun independently
# after the setup and helper cells; its seed is fixed by pathway number 2.
dataset_verify_pathway(
    target_name=WEIGHT,
    selected_methods={"density": "geldsetzer"},
    pathway_number=2,
)

**Successful set: 4,187 / 14,776 slabs.** Complete indices and IDs are stored under `('slab_weight_shear', ('geldsetzer', None, None))` in `dataset_successful_indices` and `dataset_successful_ids`.

selection,dataset index,pit ID,slab ID,layers,target successful in full scan
successful witness,11629,63497,63497_slab_0,3,True
unfiltered 1,129,10484,10484_slab_0,3,False
unfiltered 2,12928,69207,69207_slab_0,6,False
unfiltered 3,826,13701,13701_slab_0,2,True
unfiltered 4,23,1007,1007_slab_0,1,False
unfiltered 5,706,13085,13085_slab_0,3,False
unfiltered 6,9205,55277,55277_slab_0,4,True
unfiltered 7,13383,71419,71419_slab_0,5,False
unfiltered 8,12460,67306,67306_slab_0,1,True
unfiltered 9,8701,52745,52745_slab_0,2,True


Verified 11 selected slabs; 130 parameter comparisons; all values, standard deviations, trace outputs and availability checks agree.


### 4.3. Kim Table 2 → slab weight → shear weight

The successful witness and ten unfiltered draws use this density method. Expand a selected slab to inspect its original measurements, numerical substitutions, uncertainties, comparisons, and any failure traces.

In [340]:
# Select this weight pathway explicitly. The section can be rerun independently
# after the setup and helper cells; its seed is fixed by pathway number 3.
dataset_verify_pathway(
    target_name=WEIGHT,
    selected_methods={"density": "kim_jamieson_table2"},
    pathway_number=3,
)

**Successful set: 5,470 / 14,776 slabs.** Complete indices and IDs are stored under `('slab_weight_shear', ('kim_jamieson_table2', None, None))` in `dataset_successful_indices` and `dataset_successful_ids`.

selection,dataset index,pit ID,slab ID,layers,target successful in full scan
successful witness,13379,71398,71398_slab_0,1,True
unfiltered 1,7489,47086,47086_slab_0,4,True
unfiltered 2,3168,26748,26748_slab_0,3,True
unfiltered 3,9418,56083,56083_slab_0,2,True
unfiltered 4,14402,7794,7794_slab_0,2,False
unfiltered 5,11563,63251,63251_slab_0,6,True
unfiltered 6,5372,36418,36418_slab_0,3,False
unfiltered 7,7759,48239,48239_slab_1,4,False
unfiltered 8,1588,17780,17780_slab_0,3,True
unfiltered 9,6798,45026,45026_slab_0,1,True


Verified 11 selected slabs; 136 parameter comparisons; all values, standard deviations, trace outputs and availability checks agree.


### 4.4. Kim Table 6 → slab weight → shear weight

The successful witness and ten unfiltered draws use this density method. Expand a selected slab to inspect its original measurements, numerical substitutions, uncertainties, comparisons, and any failure traces.

In [341]:
# Select this weight pathway explicitly. The section can be rerun independently
# after the setup and helper cells; its seed is fixed by pathway number 4.
dataset_verify_pathway(
    target_name=WEIGHT,
    selected_methods={"density": "kim_jamieson_table6"},
    pathway_number=4,
)

**Successful set: 697 / 14,776 slabs.** Complete indices and IDs are stored under `('slab_weight_shear', ('kim_jamieson_table6', None, None))` in `dataset_successful_indices` and `dataset_successful_ids`.

selection,dataset index,pit ID,slab ID,layers,target successful in full scan
successful witness,11181,61499,61499_slab_0,2,True
unfiltered 1,2864,24114,24114_slab_0,5,False
unfiltered 2,11100,61164,61164_slab_1,5,False
unfiltered 3,12585,67992,67992_slab_1,3,False
unfiltered 4,2449,21807,21807_slab_0,9,False
unfiltered 5,13227,70658,70658_slab_0,1,False
unfiltered 6,7890,48664,48664_slab_0,5,False
unfiltered 7,9683,56693,56693_slab_1,4,False
unfiltered 8,8321,50722,50722_slab_0,6,False
unfiltered 9,5418,36551,36551_slab_1,6,False


Verified 11 selected slabs; 169 parameter comparisons; all values, standard deviations, trace outputs and availability checks agree.


## 5. Weight-with-elasticity pathways — actual slabs

### 5.1. Measured density

#### 5.1.1. Measured density → Bergfeld E → Köchle ν → elasticity target

One successful witness is selected if available; ten draws are made from the full dataset.

In [342]:
# Explicitly choose density, E and ν for this pathway. No manual result is
# supplied to the framework; both sides start with the selected slab's observations.
dataset_verify_pathway(
    target_name=ELASTICITY,
    selected_methods={"density": "data_flow", "elastic_modulus": "bergfeld",
                      "poissons_ratio": "kochle"},
    pathway_number=5,
)

**Successful set: 6 / 14,776 slabs.** Complete indices and IDs are stored under `('slab_weight_shear_with_elasticity', ('data_flow', 'bergfeld', 'kochle'))` in `dataset_successful_indices` and `dataset_successful_ids`.

selection,dataset index,pit ID,slab ID,layers,target successful in full scan
successful witness,6216,41375,41375_slab_1,3,True
unfiltered 1,3251,27020,27020_slab_0,2,False
unfiltered 2,7213,46244,46244_slab_1,9,False
unfiltered 3,7880,48633,48633_slab_0,4,False
unfiltered 4,171,10694,10694_slab_0,4,False
unfiltered 5,13434,7162,7162_slab_0,3,False
unfiltered 6,13494,71890,71890_slab_0,4,False
unfiltered 7,660,12905,12905_slab_1,3,False
unfiltered 8,489,12192,12192_slab_0,4,False
unfiltered 9,9994,57555,57555_slab_0,3,False


Verified 11 selected slabs; 201 parameter comparisons; all values, standard deviations, trace outputs and availability checks agree.


#### 5.1.2. Measured density → Bergfeld E → Srivastava ν → elasticity target

One successful witness is selected if available; ten draws are made from the full dataset.

In [343]:
# Explicitly choose density, E and ν for this pathway. No manual result is
# supplied to the framework; both sides start with the selected slab's observations.
dataset_verify_pathway(
    target_name=ELASTICITY,
    selected_methods={"density": "data_flow", "elastic_modulus": "bergfeld",
                      "poissons_ratio": "srivastava"},
    pathway_number=6,
)

**Successful set: 8 / 14,776 slabs.** Complete indices and IDs are stored under `('slab_weight_shear_with_elasticity', ('data_flow', 'bergfeld', 'srivastava'))` in `dataset_successful_indices` and `dataset_successful_ids`.

selection,dataset index,pit ID,slab ID,layers,target successful in full scan
successful witness,1577,17716,17716_slab_0,2,True
unfiltered 1,11931,64938,64938_slab_0,2,False
unfiltered 2,9186,5520,5520_slab_0,3,False
unfiltered 3,8434,51432,51432_slab_0,3,False
unfiltered 4,10877,60451,60451_slab_0,4,False
unfiltered 5,11158,61406,61406_slab_0,2,False
unfiltered 6,6720,44587,44587_slab_0,5,False
unfiltered 7,8974,54045,54045_slab_0,1,False
unfiltered 8,4857,33379,33379_slab_0,4,False
unfiltered 9,7275,46428,46428_slab_0,2,False


Verified 11 selected slabs; 161 parameter comparisons; all values, standard deviations, trace outputs and availability checks agree.


#### 5.1.3. Measured density → Köchle E → Köchle ν → elasticity target

One successful witness is selected if available; ten draws are made from the full dataset.

In [344]:
# Explicitly choose density, E and ν for this pathway. No manual result is
# supplied to the framework; both sides start with the selected slab's observations.
dataset_verify_pathway(
    target_name=ELASTICITY,
    selected_methods={"density": "data_flow", "elastic_modulus": "kochle",
                      "poissons_ratio": "kochle"},
    pathway_number=7,
)

**Successful set: 16 / 14,776 slabs.** Complete indices and IDs are stored under `('slab_weight_shear_with_elasticity', ('data_flow', 'kochle', 'kochle'))` in `dataset_successful_indices` and `dataset_successful_ids`.

selection,dataset index,pit ID,slab ID,layers,target successful in full scan
successful witness,9803,57030,57030_slab_0,1,True
unfiltered 1,496,12218,12218_slab_0,3,False
unfiltered 2,7173,46114,46114_slab_0,3,False
unfiltered 3,2993,24733,24733_slab_0,1,False
unfiltered 4,10870,60435,60435_slab_0,3,False
unfiltered 5,1991,19333,19333_slab_0,3,False
unfiltered 6,7938,48805,48805_slab_0,2,False
unfiltered 7,14618,8751,8751_slab_0,2,False
unfiltered 8,12994,69414,69414_slab_0,4,False
unfiltered 9,8577,52057,52057_slab_0,4,False


Verified 11 selected slabs; 145 parameter comparisons; all values, standard deviations, trace outputs and availability checks agree.


#### 5.1.4. Measured density → Köchle E → Srivastava ν → elasticity target

One successful witness is selected if available; ten draws are made from the full dataset.

In [345]:
# Explicitly choose density, E and ν for this pathway. No manual result is
# supplied to the framework; both sides start with the selected slab's observations.
dataset_verify_pathway(
    target_name=ELASTICITY,
    selected_methods={"density": "data_flow", "elastic_modulus": "kochle",
                      "poissons_ratio": "srivastava"},
    pathway_number=8,
)

**Successful set: 5 / 14,776 slabs.** Complete indices and IDs are stored under `('slab_weight_shear_with_elasticity', ('data_flow', 'kochle', 'srivastava'))` in `dataset_successful_indices` and `dataset_successful_ids`.

selection,dataset index,pit ID,slab ID,layers,target successful in full scan
successful witness,8082,49415,49415_slab_0,1,True
unfiltered 1,5397,36501,36501_slab_0,5,False
unfiltered 2,12795,68681,68681_slab_1,5,False
unfiltered 3,1044,14586,14586_slab_1,1,False
unfiltered 4,14147,74772,74772_slab_1,3,False
unfiltered 5,7492,47087,47087_slab_2,3,False
unfiltered 6,8230,5010,5010_slab_0,2,False
unfiltered 7,8929,53859,53859_slab_1,2,False
unfiltered 8,12962,69323,69323_slab_2,1,False
unfiltered 9,1893,18937,18937_slab_0,1,False


Verified 11 selected slabs; 149 parameter comparisons; all values, standard deviations, trace outputs and availability checks agree.


#### 5.1.5. Measured density → Wautier E → Köchle ν → elasticity target

One successful witness is selected if available; ten draws are made from the full dataset.

In [346]:
# Explicitly choose density, E and ν for this pathway. No manual result is
# supplied to the framework; both sides start with the selected slab's observations.
dataset_verify_pathway(
    target_name=ELASTICITY,
    selected_methods={"density": "data_flow", "elastic_modulus": "wautier",
                      "poissons_ratio": "kochle"},
    pathway_number=9,
)

**Successful set: 17 / 14,776 slabs.** Complete indices and IDs are stored under `('slab_weight_shear_with_elasticity', ('data_flow', 'wautier', 'kochle'))` in `dataset_successful_indices` and `dataset_successful_ids`.

selection,dataset index,pit ID,slab ID,layers,target successful in full scan
successful witness,10082,57837,57837_slab_0,4,True
unfiltered 1,11851,64550,64550_slab_0,3,False
unfiltered 2,1970,19261,19261_slab_0,4,False
unfiltered 3,3472,27621,27621_slab_0,8,False
unfiltered 4,7701,47906,47906_slab_0,2,False
unfiltered 5,8835,53460,53460_slab_0,5,False
unfiltered 6,4794,33049,33049_slab_0,9,False
unfiltered 7,4371,31089,31089_slab_0,3,False
unfiltered 8,11739,64044,64044_slab_0,8,False
unfiltered 9,453,12056,12056_slab_0,3,False


Verified 11 selected slabs; 253 parameter comparisons; all values, standard deviations, trace outputs and availability checks agree.


#### 5.1.6. Measured density → Wautier E → Srivastava ν → elasticity target

One successful witness is selected if available; ten draws are made from the full dataset.

In [347]:
# Explicitly choose density, E and ν for this pathway. No manual result is
# supplied to the framework; both sides start with the selected slab's observations.
dataset_verify_pathway(
    target_name=ELASTICITY,
    selected_methods={"density": "data_flow", "elastic_modulus": "wautier",
                      "poissons_ratio": "srivastava"},
    pathway_number=10,
)

**Successful set: 12 / 14,776 slabs.** Complete indices and IDs are stored under `('slab_weight_shear_with_elasticity', ('data_flow', 'wautier', 'srivastava'))` in `dataset_successful_indices` and `dataset_successful_ids`.

selection,dataset index,pit ID,slab ID,layers,target successful in full scan
successful witness,8082,49415,49415_slab_0,1,True
unfiltered 1,12208,66384,66384_slab_1,3,False
unfiltered 2,10679,59881,59881_slab_0,8,False
unfiltered 3,13402,7148,7148_slab_0,1,False
unfiltered 4,5140,35529,35529_slab_0,3,False
unfiltered 5,3483,27653,27653_slab_0,5,False
unfiltered 6,7886,48644,48644_slab_1,6,False
unfiltered 7,1021,14524,14524_slab_0,4,False
unfiltered 8,10661,59830,59830_slab_0,6,False
unfiltered 9,14079,74472,74472_slab_0,2,False


Verified 11 selected slabs; 201 parameter comparisons; all values, standard deviations, trace outputs and availability checks agree.


#### 5.1.7. Measured density → Schöttner E → Köchle ν → elasticity target

One successful witness is selected if available; ten draws are made from the full dataset.

In [348]:
# Explicitly choose density, E and ν for this pathway. No manual result is
# supplied to the framework; both sides start with the selected slab's observations.
dataset_verify_pathway(
    target_name=ELASTICITY,
    selected_methods={"density": "data_flow", "elastic_modulus": "schottner",
                      "poissons_ratio": "kochle"},
    pathway_number=11,
)

**Successful set: 17 / 14,776 slabs.** Complete indices and IDs are stored under `('slab_weight_shear_with_elasticity', ('data_flow', 'schottner', 'kochle'))` in `dataset_successful_indices` and `dataset_successful_ids`.

selection,dataset index,pit ID,slab ID,layers,target successful in full scan
successful witness,12664,68248,68248_slab_0,1,True
unfiltered 1,9508,56253,56253_slab_0,5,False
unfiltered 2,13955,73797,73797_slab_0,4,False
unfiltered 3,177,10756,10756_slab_0,1,False
unfiltered 4,6245,41558,41558_slab_0,4,False
unfiltered 5,6625,43799,43799_slab_0,1,False
unfiltered 6,5146,35535,35535_slab_1,5,False
unfiltered 7,10455,59265,59265_slab_0,5,False
unfiltered 8,10536,59495,59495_slab_0,7,False
unfiltered 9,3820,28924,28924_slab_0,4,False


Verified 11 selected slabs; 205 parameter comparisons; all values, standard deviations, trace outputs and availability checks agree.


#### 5.1.8. Measured density → Schöttner E → Srivastava ν → elasticity target

One successful witness is selected if available; ten draws are made from the full dataset.

In [349]:
# Explicitly choose density, E and ν for this pathway. No manual result is
# supplied to the framework; both sides start with the selected slab's observations.
dataset_verify_pathway(
    target_name=ELASTICITY,
    selected_methods={"density": "data_flow", "elastic_modulus": "schottner",
                      "poissons_ratio": "srivastava"},
    pathway_number=12,
)

**Successful set: 12 / 14,776 slabs.** Complete indices and IDs are stored under `('slab_weight_shear_with_elasticity', ('data_flow', 'schottner', 'srivastava'))` in `dataset_successful_indices` and `dataset_successful_ids`.

selection,dataset index,pit ID,slab ID,layers,target successful in full scan
successful witness,12176,66174,66174_slab_0,1,True
unfiltered 1,5777,38450,38450_slab_0,2,False
unfiltered 2,9491,56215,56215_slab_0,3,False
unfiltered 3,4635,32144,32144_slab_0,4,False
unfiltered 4,13990,73945,73945_slab_0,2,False
unfiltered 5,2827,24004,24004_slab_0,1,False
unfiltered 6,794,13403,13403_slab_0,5,False
unfiltered 7,13494,71890,71890_slab_0,4,False
unfiltered 8,3958,29539,29539_slab_2,9,False
unfiltered 9,3775,28724,28724_slab_3,5,False


Verified 11 selected slabs; 181 parameter comparisons; all values, standard deviations, trace outputs and availability checks agree.


### 5.2. Geldsetzer density

#### 5.2.1. Geldsetzer → Bergfeld E → Köchle ν → elasticity target

One successful witness is selected if available; ten draws are made from the full dataset.

In [350]:
# Explicitly choose density, E and ν for this pathway. No manual result is
# supplied to the framework; both sides start with the selected slab's observations.
dataset_verify_pathway(
    target_name=ELASTICITY,
    selected_methods={"density": "geldsetzer", "elastic_modulus": "bergfeld",
                      "poissons_ratio": "kochle"},
    pathway_number=13,
)

**Successful set: 443 / 14,776 slabs.** Complete indices and IDs are stored under `('slab_weight_shear_with_elasticity', ('geldsetzer', 'bergfeld', 'kochle'))` in `dataset_successful_indices` and `dataset_successful_ids`.

selection,dataset index,pit ID,slab ID,layers,target successful in full scan
successful witness,5316,36217,36217_slab_0,4,True
unfiltered 1,4331,30910,30910_slab_0,1,False
unfiltered 2,12760,68616,68616_slab_1,3,False
unfiltered 3,3310,27159,27159_slab_0,10,False
unfiltered 4,8743,530,530_slab_0,2,False
unfiltered 5,5779,38452,38452_slab_1,4,False
unfiltered 6,3943,29503,29503_slab_0,2,False
unfiltered 7,9391,56016,56016_slab_1,3,False
unfiltered 8,6012,40006,40006_slab_1,5,False
unfiltered 9,375,11686,11686_slab_0,4,False


layer,grain,hardness,h (cm),σ h (cm),d (mm),σ d (mm),measured ρ (kg/m³),σ measured ρ (kg/m³)
1,PPrm,F+,8,0.4,None,None,None,None
2,DFbk,1F,8,0.4,None,None,None,None
3,DF,4F,2,0.1,None,None,None,None
4,DF,1F,12,0.6,None,None,None,None
5,IFrc,P+,1,0.05,None,None,None,None
6,—,P,14,0.7,None,None,None,None
7,MFcr,K-,2,0.1,None,None,None,None
8,—,1F+,23,1.15,None,None,None,None
9,RGsr,P,10,0.5,None,None,None,None
10,DH,4F,5,0.25,None,None,None,None


Verified 11 selected slabs; 248 parameter comparisons; all values, standard deviations, trace outputs and availability checks agree.


#### 5.2.2. Geldsetzer → Bergfeld E → Srivastava ν → elasticity target

One successful witness is selected if available; ten draws are made from the full dataset.

In [351]:
# Explicitly choose density, E and ν for this pathway. No manual result is
# supplied to the framework; both sides start with the selected slab's observations.
dataset_verify_pathway(
    target_name=ELASTICITY,
    selected_methods={"density": "geldsetzer", "elastic_modulus": "bergfeld",
                      "poissons_ratio": "srivastava"},
    pathway_number=14,
)

**Successful set: 349 / 14,776 slabs.** Complete indices and IDs are stored under `('slab_weight_shear_with_elasticity', ('geldsetzer', 'bergfeld', 'srivastava'))` in `dataset_successful_indices` and `dataset_successful_ids`.

selection,dataset index,pit ID,slab ID,layers,target successful in full scan
successful witness,13259,70819,70819_slab_1,3,True
unfiltered 1,2890,24285,24285_slab_0,5,False
unfiltered 2,11895,64759,64759_slab_0,2,False
unfiltered 3,1216,15445,15445_slab_0,3,False
unfiltered 4,5050,34965,34965_slab_0,3,False
unfiltered 5,10634,59749,59749_slab_1,2,False
unfiltered 6,307,11373,11373_slab_0,5,False
unfiltered 7,3136,26629,26629_slab_0,3,False
unfiltered 8,8619,52306,52306_slab_0,3,False
unfiltered 9,14195,74987,74987_slab_0,5,False


Verified 11 selected slabs; 228 parameter comparisons; all values, standard deviations, trace outputs and availability checks agree.


#### 5.2.3. Geldsetzer → Köchle E → Köchle ν → elasticity target

One successful witness is selected if available; ten draws are made from the full dataset.

In [352]:
# Explicitly choose density, E and ν for this pathway. No manual result is
# supplied to the framework; both sides start with the selected slab's observations.
dataset_verify_pathway(
    target_name=ELASTICITY,
    selected_methods={"density": "geldsetzer", "elastic_modulus": "kochle",
                      "poissons_ratio": "kochle"},
    pathway_number=15,
)

**Successful set: 677 / 14,776 slabs.** Complete indices and IDs are stored under `('slab_weight_shear_with_elasticity', ('geldsetzer', 'kochle', 'kochle'))` in `dataset_successful_indices` and `dataset_successful_ids`.

selection,dataset index,pit ID,slab ID,layers,target successful in full scan
successful witness,10008,57606,57606_slab_0,4,True
unfiltered 1,6211,41316,41316_slab_0,9,False
unfiltered 2,12786,68670,68670_slab_0,2,False
unfiltered 3,3684,28316,28316_slab_1,4,False
unfiltered 4,3208,26889,26889_slab_0,1,False
unfiltered 5,11080,61112,61112_slab_0,5,False
unfiltered 6,11786,64248,64248_slab_0,1,False
unfiltered 7,9360,55956,55956_slab_0,1,False
unfiltered 8,14402,7794,7794_slab_0,2,False
unfiltered 9,7269,46417,46417_slab_0,3,False


Verified 11 selected slabs; 213 parameter comparisons; all values, standard deviations, trace outputs and availability checks agree.


#### 5.2.4. Geldsetzer → Köchle E → Srivastava ν → elasticity target

One successful witness is selected if available; ten draws are made from the full dataset.

In [353]:
# Explicitly choose density, E and ν for this pathway. No manual result is
# supplied to the framework; both sides start with the selected slab's observations.
dataset_verify_pathway(
    target_name=ELASTICITY,
    selected_methods={"density": "geldsetzer", "elastic_modulus": "kochle",
                      "poissons_ratio": "srivastava"},
    pathway_number=16,
)

**Successful set: 357 / 14,776 slabs.** Complete indices and IDs are stored under `('slab_weight_shear_with_elasticity', ('geldsetzer', 'kochle', 'srivastava'))` in `dataset_successful_indices` and `dataset_successful_ids`.

selection,dataset index,pit ID,slab ID,layers,target successful in full scan
successful witness,2728,23315,23315_slab_0,1,True
unfiltered 1,13105,69931,69931_slab_0,3,False
unfiltered 2,3729,28538,28538_slab_0,2,False
unfiltered 3,11024,6091,6091_slab_0,1,True
unfiltered 4,2178,20241,20241_slab_0,2,False
unfiltered 5,13469,71756,71756_slab_1,5,False
unfiltered 6,7963,48904,48904_slab_0,4,False
unfiltered 7,6626,438,438_slab_0,3,False
unfiltered 8,8879,53656,53656_slab_0,4,False
unfiltered 9,1467,1687,1687_slab_0,4,False


Verified 11 selected slabs; 183 parameter comparisons; all values, standard deviations, trace outputs and availability checks agree.


#### 5.2.5. Geldsetzer → Wautier E → Köchle ν → elasticity target

One successful witness is selected if available; ten draws are made from the full dataset.

In [354]:
# Explicitly choose density, E and ν for this pathway. No manual result is
# supplied to the framework; both sides start with the selected slab's observations.
dataset_verify_pathway(
    target_name=ELASTICITY,
    selected_methods={"density": "geldsetzer", "elastic_modulus": "wautier",
                      "poissons_ratio": "kochle"},
    pathway_number=17,
)

**Successful set: 684 / 14,776 slabs.** Complete indices and IDs are stored under `('slab_weight_shear_with_elasticity', ('geldsetzer', 'wautier', 'kochle'))` in `dataset_successful_indices` and `dataset_successful_ids`.

selection,dataset index,pit ID,slab ID,layers,target successful in full scan
successful witness,3751,28630,28630_slab_0,3,True
unfiltered 1,817,13601,13601_slab_0,7,False
unfiltered 2,13241,70729,70729_slab_0,3,False
unfiltered 3,2120,19902,19902_slab_0,3,False
unfiltered 4,3123,26566,26566_slab_0,1,True
unfiltered 5,9805,57038,57038_slab_0,3,False
unfiltered 6,3073,25659,25659_slab_0,5,False
unfiltered 7,6690,44395,44395_slab_0,3,False
unfiltered 8,1656,17985,17985_slab_2,5,False
unfiltered 9,3724,28517,28517_slab_0,5,False


Verified 11 selected slabs; 238 parameter comparisons; all values, standard deviations, trace outputs and availability checks agree.


#### 5.2.6. Geldsetzer → Wautier E → Srivastava ν → elasticity target

One successful witness is selected if available; ten draws are made from the full dataset.

In [355]:
# Explicitly choose density, E and ν for this pathway. No manual result is
# supplied to the framework; both sides start with the selected slab's observations.
dataset_verify_pathway(
    target_name=ELASTICITY,
    selected_methods={"density": "geldsetzer", "elastic_modulus": "wautier",
                      "poissons_ratio": "srivastava"},
    pathway_number=18,
)

**Successful set: 418 / 14,776 slabs.** Complete indices and IDs are stored under `('slab_weight_shear_with_elasticity', ('geldsetzer', 'wautier', 'srivastava'))` in `dataset_successful_indices` and `dataset_successful_ids`.

selection,dataset index,pit ID,slab ID,layers,target successful in full scan
successful witness,12162,66126,66126_slab_0,1,True
unfiltered 1,9150,5502,5502_slab_1,5,False
unfiltered 2,11288,62010,62010_slab_0,2,False
unfiltered 3,11871,64651,64651_slab_1,7,False
unfiltered 4,9562,56381,56381_slab_1,4,False
unfiltered 5,744,13231,13231_slab_0,3,False
unfiltered 6,9473,56176,56176_slab_0,5,False
unfiltered 7,11201,61574,61574_slab_0,3,False
unfiltered 8,3475,27639,27639_slab_0,5,False
unfiltered 9,615,12721,12721_slab_1,4,False


Verified 11 selected slabs; 238 parameter comparisons; all values, standard deviations, trace outputs and availability checks agree.


#### 5.2.7. Geldsetzer → Schöttner E → Köchle ν → elasticity target

One successful witness is selected if available; ten draws are made from the full dataset.

In [356]:
# Explicitly choose density, E and ν for this pathway. No manual result is
# supplied to the framework; both sides start with the selected slab's observations.
dataset_verify_pathway(
    target_name=ELASTICITY,
    selected_methods={"density": "geldsetzer", "elastic_modulus": "schottner",
                      "poissons_ratio": "kochle"},
    pathway_number=19,
)

**Successful set: 684 / 14,776 slabs.** Complete indices and IDs are stored under `('slab_weight_shear_with_elasticity', ('geldsetzer', 'schottner', 'kochle'))` in `dataset_successful_indices` and `dataset_successful_ids`.

selection,dataset index,pit ID,slab ID,layers,target successful in full scan
successful witness,12322,6679,6679_slab_0,2,True
unfiltered 1,7255,46365,46365_slab_0,4,False
unfiltered 2,3696,28356,28356_slab_0,4,False
unfiltered 3,10465,59285,59285_slab_0,9,False
unfiltered 4,5717,38043,38043_slab_0,2,False
unfiltered 5,7539,47206,47206_slab_0,7,False
unfiltered 6,4644,322,322_slab_0,2,False
unfiltered 7,2967,24622,24622_slab_0,3,False
unfiltered 8,11812,64370,64370_slab_0,5,False
unfiltered 9,11223,61651,61651_slab_0,1,True


layer,grain,hardness,h (cm),σ h (cm),d (mm),σ d (mm),measured ρ (kg/m³),σ measured ρ (kg/m³)
1,PP,F,16,0.8,1,0.5,None,None
2,IFrc,P+,5,0.25,—,—,None,None
3,MFcl,4F+,15,0.75,0.5,0.5,None,None
4,MF,P-,2,0.1,—,—,None,None
5,MFcl,4F,3,0.15,1,0.5,None,None
6,MF,1F,3,0.15,—,—,None,None
7,MFcl,4F+,6,0.3,1,0.5,None,None
8,MF,P,2,0.1,—,—,None,None
9,MFcl,4F,8,0.4,0.5,0.5,None,None
10,MF,4F+,2,0.1,—,—,None,None


Verified 11 selected slabs; 283 parameter comparisons; all values, standard deviations, trace outputs and availability checks agree.


#### 5.2.8. Geldsetzer → Schöttner E → Srivastava ν → elasticity target

One successful witness is selected if available; ten draws are made from the full dataset.

In [357]:
# Explicitly choose density, E and ν for this pathway. No manual result is
# supplied to the framework; both sides start with the selected slab's observations.
dataset_verify_pathway(
    target_name=ELASTICITY,
    selected_methods={"density": "geldsetzer", "elastic_modulus": "schottner",
                      "poissons_ratio": "srivastava"},
    pathway_number=20,
)

**Successful set: 418 / 14,776 slabs.** Complete indices and IDs are stored under `('slab_weight_shear_with_elasticity', ('geldsetzer', 'schottner', 'srivastava'))` in `dataset_successful_indices` and `dataset_successful_ids`.

selection,dataset index,pit ID,slab ID,layers,target successful in full scan
successful witness,10512,59406,59406_slab_1,1,True
unfiltered 1,3600,28025,28025_slab_0,4,False
unfiltered 2,9178,55157,55157_slab_0,2,False
unfiltered 3,5468,36738,36738_slab_0,4,False
unfiltered 4,14180,74930,74930_slab_0,3,False
unfiltered 5,6747,44738,44738_slab_0,3,False
unfiltered 6,612,12709,12709_slab_0,3,False
unfiltered 7,12023,65383,65383_slab_1,3,False
unfiltered 8,12996,69418,69418_slab_0,3,False
unfiltered 9,8469,51539,51539_slab_0,7,False


Verified 11 selected slabs; 223 parameter comparisons; all values, standard deviations, trace outputs and availability checks agree.


### 5.3. Kim Table 2 density

#### 5.3.1. Kim Table 2 → Bergfeld E → Köchle ν → elasticity target

One successful witness is selected if available; ten draws are made from the full dataset.

In [358]:
# Explicitly choose density, E and ν for this pathway. No manual result is
# supplied to the framework; both sides start with the selected slab's observations.
dataset_verify_pathway(
    target_name=ELASTICITY,
    selected_methods={"density": "kim_jamieson_table2", "elastic_modulus": "bergfeld",
                      "poissons_ratio": "kochle"},
    pathway_number=21,
)

**Successful set: 465 / 14,776 slabs.** Complete indices and IDs are stored under `('slab_weight_shear_with_elasticity', ('kim_jamieson_table2', 'bergfeld', 'kochle'))` in `dataset_successful_indices` and `dataset_successful_ids`.

selection,dataset index,pit ID,slab ID,layers,target successful in full scan
successful witness,5863,39003,39003_slab_0,1,True
unfiltered 1,3963,29555,29555_slab_0,4,False
unfiltered 2,12514,6769,6769_slab_0,11,False
unfiltered 3,12023,65383,65383_slab_1,3,False
unfiltered 4,6154,41025,41025_slab_0,4,False
unfiltered 5,1192,15298,15298_slab_0,2,False
unfiltered 6,8397,51165,51165_slab_0,3,False
unfiltered 7,669,12928,12928_slab_1,3,False
unfiltered 8,3419,27467,27467_slab_0,3,False
unfiltered 9,4048,29895,29895_slab_0,2,False


layer,grain,hardness,h (cm),σ h (cm),d (mm),σ d (mm),measured ρ (kg/m³),σ measured ρ (kg/m³)
1,MFcr,K,1,0.05,—,—,None,None
2,RGsr,P,14,0.7,0.3,0.5,None,None
3,MFcr,K,1,0.05,—,—,None,None
4,MFpc,1F,22,1.1,—,—,None,None
5,MFcr,P,2,0.1,—,—,None,None
6,FCxr,4F,10,0.5,—,—,None,None
7,MFcr,K,1,0.05,—,—,None,None
8,FCxr,1F,1,0.05,0.5,0.5,None,None
9,RGsr,1F,8,0.4,0.3,0.5,None,None
10,RGsr,P,20,1,0.1,0.5,None,None


Verified 11 selected slabs; 248 parameter comparisons; all values, standard deviations, trace outputs and availability checks agree.


#### 5.3.2. Kim Table 2 → Bergfeld E → Srivastava ν → elasticity target

One successful witness is selected if available; ten draws are made from the full dataset.

In [359]:
# Explicitly choose density, E and ν for this pathway. No manual result is
# supplied to the framework; both sides start with the selected slab's observations.
dataset_verify_pathway(
    target_name=ELASTICITY,
    selected_methods={"density": "kim_jamieson_table2", "elastic_modulus": "bergfeld",
                      "poissons_ratio": "srivastava"},
    pathway_number=22,
)

**Successful set: 380 / 14,776 slabs.** Complete indices and IDs are stored under `('slab_weight_shear_with_elasticity', ('kim_jamieson_table2', 'bergfeld', 'srivastava'))` in `dataset_successful_indices` and `dataset_successful_ids`.

selection,dataset index,pit ID,slab ID,layers,target successful in full scan
successful witness,13803,7318,7318_slab_0,1,True
unfiltered 1,3455,27579,27579_slab_1,6,False
unfiltered 2,1350,16066,16066_slab_0,6,False
unfiltered 3,1872,18838,18838_slab_0,6,False
unfiltered 4,9278,55688,55688_slab_0,7,False
unfiltered 5,12208,66384,66384_slab_1,3,False
unfiltered 6,2457,21877,21877_slab_0,4,False
unfiltered 7,7679,4779,4779_slab_0,2,False
unfiltered 8,10614,59700,59700_slab_0,5,False
unfiltered 9,822,13633,13633_slab_0,4,False


Verified 11 selected slabs; 273 parameter comparisons; all values, standard deviations, trace outputs and availability checks agree.


#### 5.3.3. Kim Table 2 → Köchle E → Köchle ν → elasticity target

One successful witness is selected if available; ten draws are made from the full dataset.

In [360]:
# Explicitly choose density, E and ν for this pathway. No manual result is
# supplied to the framework; both sides start with the selected slab's observations.
dataset_verify_pathway(
    target_name=ELASTICITY,
    selected_methods={"density": "kim_jamieson_table2", "elastic_modulus": "kochle",
                      "poissons_ratio": "kochle"},
    pathway_number=23,
)

**Successful set: 581 / 14,776 slabs.** Complete indices and IDs are stored under `('slab_weight_shear_with_elasticity', ('kim_jamieson_table2', 'kochle', 'kochle'))` in `dataset_successful_indices` and `dataset_successful_ids`.

selection,dataset index,pit ID,slab ID,layers,target successful in full scan
successful witness,14653,8976,8976_slab_0,3,True
unfiltered 1,12763,6862,6862_slab_1,3,False
unfiltered 2,5170,35611,35611_slab_0,1,False
unfiltered 3,9611,565,565_slab_0,12,False
unfiltered 4,4427,31283,31283_slab_0,3,False
unfiltered 5,14329,75935,75935_slab_0,4,False
unfiltered 6,2525,22236,22236_slab_0,4,False
unfiltered 7,7242,46320,46320_slab_0,2,False
unfiltered 8,1464,16854,16854_slab_0,3,False
unfiltered 9,10987,60787,60787_slab_0,3,False


layer,grain,hardness,h (cm),σ h (cm),d (mm),σ d (mm),measured ρ (kg/m³),σ measured ρ (kg/m³)
1,PPsd,F,2,0.1,1,0.5,None,None
2,RG,F,10,0.5,0.5,0.5,None,None
3,DFbk,F,5,0.25,0.5,0.5,None,None
4,DFbk,1F,21,1.05,0.5,0.5,None,None
5,PPgp,P-,1,0.05,3,0.5,None,None
6,RGxf,P,42,2.1,0.5,0.5,None,None
7,PPgp,P-,2,0.1,2,0.5,None,None
8,PPgp,P,6,0.3,2,0.5,None,None
9,PPgp,4F,3,0.15,2,0.5,None,None
10,RGlr,P+,8,0.4,0.5,0.5,None,None


Verified 11 selected slabs; 233 parameter comparisons; all values, standard deviations, trace outputs and availability checks agree.


#### 5.3.4. Kim Table 2 → Köchle E → Srivastava ν → elasticity target

One successful witness is selected if available; ten draws are made from the full dataset.

In [361]:
# Explicitly choose density, E and ν for this pathway. No manual result is
# supplied to the framework; both sides start with the selected slab's observations.
dataset_verify_pathway(
    target_name=ELASTICITY,
    selected_methods={"density": "kim_jamieson_table2", "elastic_modulus": "kochle",
                      "poissons_ratio": "srivastava"},
    pathway_number=24,
)

**Successful set: 353 / 14,776 slabs.** Complete indices and IDs are stored under `('slab_weight_shear_with_elasticity', ('kim_jamieson_table2', 'kochle', 'srivastava'))` in `dataset_successful_indices` and `dataset_successful_ids`.

selection,dataset index,pit ID,slab ID,layers,target successful in full scan
successful witness,2141,20027,20027_slab_0,1,True
unfiltered 1,9779,56969,56969_slab_0,4,False
unfiltered 2,4725,32702,32702_slab_1,5,False
unfiltered 3,11329,62243,62243_slab_0,7,False
unfiltered 4,6036,4013,4013_slab_0,1,False
unfiltered 5,14607,8667,8667_slab_0,2,False
unfiltered 6,3684,28316,28316_slab_1,4,False
unfiltered 7,3132,26617,26617_slab_0,5,False
unfiltered 8,3916,29431,29431_slab_3,10,False
unfiltered 9,5618,37429,37429_slab_0,11,False


layer,grain,hardness,h (cm),σ h (cm),d (mm),σ d (mm),measured ρ (kg/m³),σ measured ρ (kg/m³)
1,PPsd,F,10,0.5,1,0.5,None,None
2,DFdc,F+,9,0.45,1.5,0.5,None,None
3,RGsr,4F,6,0.3,0.5,0.5,None,None
4,SHsu,F,2,0.1,6,0.5,None,None
5,RGsr,4F,27,1.35,1,0.5,None,None
6,FCsf,4F,1,0.05,1,0.5,None,None
7,FCxr,1F,9,0.45,1,0.5,None,None
8,FCsf,4F,1,0.05,1,0.5,None,None
9,FCxr,1F+,10,0.5,0.5,0.5,None,None
10,FCxr,1F,17,0.85,1,0.5,None,None


layer,grain,hardness,h (cm),σ h (cm),d (mm),σ d (mm),measured ρ (kg/m³),σ measured ρ (kg/m³)
1,PP,F,2,0.1,1,0.5,None,None
2,MFcr,P,4,0.2,1,0.5,None,None
3,RGxf,4F,1,0.05,2,0.5,None,None
4,IF,I,4,0.2,—,—,None,None
5,RGxf,1F,23,1.15,0.5,0.5,None,None
6,PPgp,4F,1,0.05,2,0.5,None,None
7,RGxf,1F,12,0.6,0.5,0.5,None,None
8,RGxf,—,22,1.1,0.5,0.5,None,None
9,RGxf,4F+,19,0.95,0.5,0.5,None,None
10,RGxf,4F,18,0.9,0.5,0.5,None,None


Verified 11 selected slabs; 293 parameter comparisons; all values, standard deviations, trace outputs and availability checks agree.


#### 5.3.5. Kim Table 2 → Wautier E → Köchle ν → elasticity target

One successful witness is selected if available; ten draws are made from the full dataset.

In [362]:
# Explicitly choose density, E and ν for this pathway. No manual result is
# supplied to the framework; both sides start with the selected slab's observations.
dataset_verify_pathway(
    target_name=ELASTICITY,
    selected_methods={"density": "kim_jamieson_table2", "elastic_modulus": "wautier",
                      "poissons_ratio": "kochle"},
    pathway_number=25,
)

**Successful set: 687 / 14,776 slabs.** Complete indices and IDs are stored under `('slab_weight_shear_with_elasticity', ('kim_jamieson_table2', 'wautier', 'kochle'))` in `dataset_successful_indices` and `dataset_successful_ids`.

selection,dataset index,pit ID,slab ID,layers,target successful in full scan
successful witness,8922,53835,53835_slab_1,3,True
unfiltered 1,3316,27189,27189_slab_1,1,False
unfiltered 2,14656,900,900_slab_0,6,False
unfiltered 3,294,11347,11347_slab_0,1,False
unfiltered 4,12864,68982,68982_slab_0,3,False
unfiltered 5,9491,56215,56215_slab_0,3,False
unfiltered 6,9499,56233,56233_slab_1,6,False
unfiltered 7,13454,71674,71674_slab_0,6,False
unfiltered 8,11831,64448,64448_slab_1,4,False
unfiltered 9,8177,49831,49831_slab_1,3,False


Verified 11 selected slabs; 238 parameter comparisons; all values, standard deviations, trace outputs and availability checks agree.


#### 5.3.6. Kim Table 2 → Wautier E → Srivastava ν → elasticity target

One successful witness is selected if available; ten draws are made from the full dataset.

In [363]:
# Explicitly choose density, E and ν for this pathway. No manual result is
# supplied to the framework; both sides start with the selected slab's observations.
dataset_verify_pathway(
    target_name=ELASTICITY,
    selected_methods={"density": "kim_jamieson_table2", "elastic_modulus": "wautier",
                      "poissons_ratio": "srivastava"},
    pathway_number=26,
)

**Successful set: 415 / 14,776 slabs.** Complete indices and IDs are stored under `('slab_weight_shear_with_elasticity', ('kim_jamieson_table2', 'wautier', 'srivastava'))` in `dataset_successful_indices` and `dataset_successful_ids`.

selection,dataset index,pit ID,slab ID,layers,target successful in full scan
successful witness,3201,26857,26857_slab_0,3,True
unfiltered 1,6634,43881,43881_slab_0,4,False
unfiltered 2,8754,53036,53036_slab_0,1,False
unfiltered 3,251,11036,11036_slab_1,1,False
unfiltered 4,1567,17693,17693_slab_0,4,False
unfiltered 5,5870,39054,39054_slab_0,5,False
unfiltered 6,3256,2703,2703_slab_0,2,False
unfiltered 7,10257,58563,58563_slab_0,5,False
unfiltered 8,8892,537,537_slab_1,7,False
unfiltered 9,7360,46682,46682_slab_0,1,False


Verified 11 selected slabs; 218 parameter comparisons; all values, standard deviations, trace outputs and availability checks agree.


#### 5.3.7. Kim Table 2 → Schöttner E → Köchle ν → elasticity target

One successful witness is selected if available; ten draws are made from the full dataset.

In [364]:
# Explicitly choose density, E and ν for this pathway. No manual result is
# supplied to the framework; both sides start with the selected slab's observations.
dataset_verify_pathway(
    target_name=ELASTICITY,
    selected_methods={"density": "kim_jamieson_table2", "elastic_modulus": "schottner",
                      "poissons_ratio": "kochle"},
    pathway_number=27,
)

**Successful set: 687 / 14,776 slabs.** Complete indices and IDs are stored under `('slab_weight_shear_with_elasticity', ('kim_jamieson_table2', 'schottner', 'kochle'))` in `dataset_successful_indices` and `dataset_successful_ids`.

selection,dataset index,pit ID,slab ID,layers,target successful in full scan
successful witness,11498,62999,62999_slab_0,2,True
unfiltered 1,1568,17694,17694_slab_0,2,False
unfiltered 2,6067,40447,40447_slab_0,1,False
unfiltered 3,857,13863,13863_slab_2,4,False
unfiltered 4,7361,46687,46687_slab_0,2,False
unfiltered 5,2351,2120,2120_slab_0,3,False
unfiltered 6,6080,4055,4055_slab_0,1,True
unfiltered 7,3865,29154,29154_slab_0,5,False
unfiltered 8,1149,15069,15069_slab_0,6,False
unfiltered 9,7907,4870,4870_slab_0,4,False


Verified 11 selected slabs; 203 parameter comparisons; all values, standard deviations, trace outputs and availability checks agree.


#### 5.3.8. Kim Table 2 → Schöttner E → Srivastava ν → elasticity target

One successful witness is selected if available; ten draws are made from the full dataset.

In [365]:
# Explicitly choose density, E and ν for this pathway. No manual result is
# supplied to the framework; both sides start with the selected slab's observations.
dataset_verify_pathway(
    target_name=ELASTICITY,
    selected_methods={"density": "kim_jamieson_table2", "elastic_modulus": "schottner",
                      "poissons_ratio": "srivastava"},
    pathway_number=28,
)

**Successful set: 415 / 14,776 slabs.** Complete indices and IDs are stored under `('slab_weight_shear_with_elasticity', ('kim_jamieson_table2', 'schottner', 'srivastava'))` in `dataset_successful_indices` and `dataset_successful_ids`.

selection,dataset index,pit ID,slab ID,layers,target successful in full scan
successful witness,5553,37085,37085_slab_1,1,True
unfiltered 1,2574,22545,22545_slab_0,5,False
unfiltered 2,12532,6778,6778_slab_0,3,False
unfiltered 3,2876,24210,24210_slab_0,6,False
unfiltered 4,6951,45521,45521_slab_0,6,False
unfiltered 5,13602,72349,72349_slab_0,2,False
unfiltered 6,3728,28535,28535_slab_0,4,False
unfiltered 7,13093,69865,69865_slab_0,4,False
unfiltered 8,5451,36683,36683_slab_0,3,False
unfiltered 9,350,11551,11551_slab_1,4,False


Verified 11 selected slabs; 228 parameter comparisons; all values, standard deviations, trace outputs and availability checks agree.


### 5.4. Kim Table 6 density

#### 5.4.1. Kim Table 6 → Bergfeld E → Köchle ν → elasticity target

One successful witness is selected if available; ten draws are made from the full dataset.

This combination has an empty intersection of supported grain classes: Kim Table 6 ∩ Bergfeld admits PP/DF, whereas Köchle ν admits RG/FC/DH. The full-dataset scan must therefore find zero successful slabs. The unfiltered examples verify the available upstream steps and the failed target.

In [366]:
# Explicitly choose density, E and ν for this pathway. No manual result is
# supplied to the framework; both sides start with the selected slab's observations.
dataset_verify_pathway(
    target_name=ELASTICITY,
    selected_methods={"density": "kim_jamieson_table6", "elastic_modulus": "bergfeld",
                      "poissons_ratio": "kochle"},
    pathway_number=29,
)

**Successful set: 0 / 14,776 slabs.** Complete indices and IDs are stored under `('slab_weight_shear_with_elasticity', ('kim_jamieson_table6', 'bergfeld', 'kochle'))` in `dataset_successful_indices` and `dataset_successful_ids`.

**No successful slab exists in this dataset for this pathway.** The ten unfiltered examples below still verify available upstream steps and failures.

selection,dataset index,pit ID,slab ID,layers,target successful in full scan
unfiltered 1,9740,56837,56837_slab_0,6,False
unfiltered 2,10432,59187,59187_slab_0,5,False
unfiltered 3,2736,2333,2333_slab_0,6,False
unfiltered 4,14065,74398,74398_slab_0,4,False
unfiltered 5,14189,7496,7496_slab_0,2,False
unfiltered 6,5674,37699,37699_slab_0,5,False
unfiltered 7,14732,9726,9726_slab_0,3,False
unfiltered 8,9097,54775,54775_slab_0,4,False
unfiltered 9,4422,31273,31273_slab_0,5,False
unfiltered 10,13863,7347,7347_slab_0,2,False


Verified 10 selected slabs; 240 parameter comparisons; all values, standard deviations, trace outputs and availability checks agree.


#### 5.4.2. Kim Table 6 → Bergfeld E → Srivastava ν → elasticity target

One successful witness is selected if available; ten draws are made from the full dataset.

This pathway has no successful slab in the observed cohort. An empty dataset success set alone does not prove that all possible inputs are incompatible. The ten unfiltered examples verify available upstream values and the failed target.

In [367]:
# Explicitly choose density, E and ν for this pathway. No manual result is
# supplied to the framework; both sides start with the selected slab's observations.
dataset_verify_pathway(
    target_name=ELASTICITY,
    selected_methods={"density": "kim_jamieson_table6", "elastic_modulus": "bergfeld",
                      "poissons_ratio": "srivastava"},
    pathway_number=30,
)

**Successful set: 0 / 14,776 slabs.** Complete indices and IDs are stored under `('slab_weight_shear_with_elasticity', ('kim_jamieson_table6', 'bergfeld', 'srivastava'))` in `dataset_successful_indices` and `dataset_successful_ids`.

**No successful slab exists in this dataset for this pathway.** The ten unfiltered examples below still verify available upstream steps and failures.

selection,dataset index,pit ID,slab ID,layers,target successful in full scan
unfiltered 1,13350,71272,71272_slab_0,3,False
unfiltered 2,9785,56983,56983_slab_0,1,False
unfiltered 3,1143,15031,15031_slab_0,6,False
unfiltered 4,8619,52306,52306_slab_0,3,False
unfiltered 5,12740,68552,68552_slab_1,5,False
unfiltered 6,7680,47790,47790_slab_0,7,False
unfiltered 7,8486,5160,5160_slab_0,4,False
unfiltered 8,5810,38602,38602_slab_0,6,False
unfiltered 9,10702,59957,59957_slab_0,2,False
unfiltered 10,230,10949,10949_slab_0,3,False


Verified 10 selected slabs; 230 parameter comparisons; all values, standard deviations, trace outputs and availability checks agree.


#### 5.4.3. Kim Table 6 → Köchle E → Köchle ν → elasticity target

One successful witness is selected if available; ten draws are made from the full dataset.

In [368]:
# Explicitly choose density, E and ν for this pathway. No manual result is
# supplied to the framework; both sides start with the selected slab's observations.
dataset_verify_pathway(
    target_name=ELASTICITY,
    selected_methods={"density": "kim_jamieson_table6", "elastic_modulus": "kochle",
                      "poissons_ratio": "kochle"},
    pathway_number=31,
)

**Successful set: 6 / 14,776 slabs.** Complete indices and IDs are stored under `('slab_weight_shear_with_elasticity', ('kim_jamieson_table6', 'kochle', 'kochle'))` in `dataset_successful_indices` and `dataset_successful_ids`.

selection,dataset index,pit ID,slab ID,layers,target successful in full scan
successful witness,9785,56983,56983_slab_0,1,True
unfiltered 1,2462,21899,21899_slab_0,8,False
unfiltered 2,14456,8059,8059_slab_1,3,False
unfiltered 3,6911,4539,4539_slab_0,2,False
unfiltered 4,3716,28469,28469_slab_0,9,False
unfiltered 5,12932,69213,69213_slab_0,7,False
unfiltered 6,4887,33610,33610_slab_0,5,False
unfiltered 7,6152,41019,41019_slab_2,2,False
unfiltered 8,9165,55079,55079_slab_0,5,False
unfiltered 9,11658,63637,63637_slab_0,3,False


Verified 11 selected slabs; 263 parameter comparisons; all values, standard deviations, trace outputs and availability checks agree.


#### 5.4.4. Kim Table 6 → Köchle E → Srivastava ν → elasticity target

One successful witness is selected if available; ten draws are made from the full dataset.

In [369]:
# Explicitly choose density, E and ν for this pathway. No manual result is
# supplied to the framework; both sides start with the selected slab's observations.
dataset_verify_pathway(
    target_name=ELASTICITY,
    selected_methods={"density": "kim_jamieson_table6", "elastic_modulus": "kochle",
                      "poissons_ratio": "srivastava"},
    pathway_number=32,
)

**Successful set: 5 / 14,776 slabs.** Complete indices and IDs are stored under `('slab_weight_shear_with_elasticity', ('kim_jamieson_table6', 'kochle', 'srivastava'))` in `dataset_successful_indices` and `dataset_successful_ids`.

selection,dataset index,pit ID,slab ID,layers,target successful in full scan
successful witness,10366,58935,58935_slab_0,1,True
unfiltered 1,14764,9963,9963_slab_0,6,False
unfiltered 2,12747,68579,68579_slab_0,7,False
unfiltered 3,10426,59155,59155_slab_0,2,False
unfiltered 4,1845,18720,18720_slab_0,5,False
unfiltered 5,6744,44714,44714_slab_0,5,False
unfiltered 6,7659,47622,47622_slab_0,3,False
unfiltered 7,11943,64970,64970_slab_0,6,False
unfiltered 8,6521,4293,4293_slab_1,3,False
unfiltered 9,14198,74992,74992_slab_0,10,False


layer,grain,hardness,h (cm),σ h (cm),d (mm),σ d (mm),measured ρ (kg/m³),σ measured ρ (kg/m³)
1,MFcr,K-,2,0.1,—,—,None,None
2,RG,F,3,0.15,—,—,None,None
3,IFil,P,1,0.05,—,—,None,None
4,RG,F,2,0.1,—,—,None,None
5,IFil,P,1,0.05,—,—,None,None
6,RG,F,6,0.3,—,—,None,None
7,IFil,P,1,0.05,—,—,None,None
8,RG,4F,14,0.7,—,—,None,None
9,RG,1F,15,0.75,—,—,None,None
10,SHsu,F,2,0.1,8,0.5,None,None


Verified 11 selected slabs; 283 parameter comparisons; all values, standard deviations, trace outputs and availability checks agree.


#### 5.4.5. Kim Table 6 → Wautier E → Köchle ν → elasticity target

One successful witness is selected if available; ten draws are made from the full dataset.

In [370]:
# Explicitly choose density, E and ν for this pathway. No manual result is
# supplied to the framework; both sides start with the selected slab's observations.
dataset_verify_pathway(
    target_name=ELASTICITY,
    selected_methods={"density": "kim_jamieson_table6", "elastic_modulus": "wautier",
                      "poissons_ratio": "kochle"},
    pathway_number=33,
)

**Successful set: 6 / 14,776 slabs.** Complete indices and IDs are stored under `('slab_weight_shear_with_elasticity', ('kim_jamieson_table6', 'wautier', 'kochle'))` in `dataset_successful_indices` and `dataset_successful_ids`.

selection,dataset index,pit ID,slab ID,layers,target successful in full scan
successful witness,9785,56983,56983_slab_0,1,True
unfiltered 1,4142,30229,30229_slab_1,7,False
unfiltered 2,12260,66590,66590_slab_0,3,False
unfiltered 3,8569,51959,51959_slab_0,4,False
unfiltered 4,10679,59881,59881_slab_0,8,False
unfiltered 5,10605,59687,59687_slab_0,8,False
unfiltered 6,14515,8284,8284_slab_0,4,False
unfiltered 7,2625,2275,2275_slab_0,3,False
unfiltered 8,11272,61949,61949_slab_0,1,False
unfiltered 9,11100,61164,61164_slab_1,5,False


Verified 11 selected slabs; 273 parameter comparisons; all values, standard deviations, trace outputs and availability checks agree.


#### 5.4.6. Kim Table 6 → Wautier E → Srivastava ν → elasticity target

One successful witness is selected if available; ten draws are made from the full dataset.

In [371]:
# Explicitly choose density, E and ν for this pathway. No manual result is
# supplied to the framework; both sides start with the selected slab's observations.
dataset_verify_pathway(
    target_name=ELASTICITY,
    selected_methods={"density": "kim_jamieson_table6", "elastic_modulus": "wautier",
                      "poissons_ratio": "srivastava"},
    pathway_number=34,
)

**Successful set: 5 / 14,776 slabs.** Complete indices and IDs are stored under `('slab_weight_shear_with_elasticity', ('kim_jamieson_table6', 'wautier', 'srivastava'))` in `dataset_successful_indices` and `dataset_successful_ids`.

selection,dataset index,pit ID,slab ID,layers,target successful in full scan
successful witness,8042,49211,49211_slab_0,1,True
unfiltered 1,337,11511,11511_slab_0,3,False
unfiltered 2,13568,72199,72199_slab_0,3,False
unfiltered 3,6530,42999,42999_slab_0,1,False
unfiltered 4,3190,26835,26835_slab_2,3,False
unfiltered 5,9382,56000,56000_slab_0,3,False
unfiltered 6,9087,54700,54700_slab_0,4,False
unfiltered 7,3833,29007,29007_slab_0,4,False
unfiltered 8,8491,51620,51620_slab_0,3,False
unfiltered 9,2719,23288,23288_slab_0,6,False


Verified 11 selected slabs; 198 parameter comparisons; all values, standard deviations, trace outputs and availability checks agree.


#### 5.4.7. Kim Table 6 → Schöttner E → Köchle ν → elasticity target

One successful witness is selected if available; ten draws are made from the full dataset.

In [372]:
# Explicitly choose density, E and ν for this pathway. No manual result is
# supplied to the framework; both sides start with the selected slab's observations.
dataset_verify_pathway(
    target_name=ELASTICITY,
    selected_methods={"density": "kim_jamieson_table6", "elastic_modulus": "schottner",
                      "poissons_ratio": "kochle"},
    pathway_number=35,
)

**Successful set: 6 / 14,776 slabs.** Complete indices and IDs are stored under `('slab_weight_shear_with_elasticity', ('kim_jamieson_table6', 'schottner', 'kochle'))` in `dataset_successful_indices` and `dataset_successful_ids`.

selection,dataset index,pit ID,slab ID,layers,target successful in full scan
successful witness,10366,58935,58935_slab_0,1,True
unfiltered 1,11553,63230,63230_slab_0,3,False
unfiltered 2,5199,35721,35721_slab_0,3,False
unfiltered 3,4068,29954,29954_slab_0,4,False
unfiltered 4,7951,48852,48852_slab_0,4,False
unfiltered 5,1969,19252,19252_slab_0,3,False
unfiltered 6,13878,73519,73519_slab_0,2,False
unfiltered 7,10597,59662,59662_slab_0,1,False
unfiltered 8,192,10792,10792_slab_1,6,False
unfiltered 9,1679,18082,18082_slab_0,3,False


Verified 11 selected slabs; 193 parameter comparisons; all values, standard deviations, trace outputs and availability checks agree.


#### 5.4.8. Kim Table 6 → Schöttner E → Srivastava ν → elasticity target

One successful witness is selected if available; ten draws are made from the full dataset.

In [373]:
# Explicitly choose density, E and ν for this pathway. No manual result is
# supplied to the framework; both sides start with the selected slab's observations.
dataset_verify_pathway(
    target_name=ELASTICITY,
    selected_methods={"density": "kim_jamieson_table6", "elastic_modulus": "schottner",
                      "poissons_ratio": "srivastava"},
    pathway_number=36,
)

**Successful set: 5 / 14,776 slabs.** Complete indices and IDs are stored under `('slab_weight_shear_with_elasticity', ('kim_jamieson_table6', 'schottner', 'srivastava'))` in `dataset_successful_indices` and `dataset_successful_ids`.

selection,dataset index,pit ID,slab ID,layers,target successful in full scan
successful witness,4043,29863,29863_slab_0,2,True
unfiltered 1,8582,52064,52064_slab_0,2,False
unfiltered 2,13932,73722,73722_slab_0,7,False
unfiltered 3,11872,64652,64652_slab_0,1,False
unfiltered 4,2599,22590,22590_slab_0,1,False
unfiltered 5,9877,5721,5721_slab_0,3,False
unfiltered 6,8982,54121,54121_slab_0,8,False
unfiltered 7,12090,65745,65745_slab_0,6,False
unfiltered 8,11351,62321,62321_slab_0,2,False
unfiltered 9,5743,38199,38199_slab_0,10,False


layer,grain,hardness,h (cm),σ h (cm),d (mm),σ d (mm),measured ρ (kg/m³),σ measured ρ (kg/m³)
1,FCsf,F,1.5,0.075,0.5,0.5,None,None
2,IFrc,K,1,0.05,—,—,None,None
3,IFrc,—,5.5,0.275,—,—,None,None
4,RG,4F+,13,0.65,—,—,None,None
5,RGxf,4F,1,0.05,—,—,None,None
6,RG,1F,16,0.8,—,—,None,None
7,RGxf,4F+,1,0.05,—,—,None,None
8,RGwp,P-,22,1.1,—,—,None,None
9,—,P,0.5,0.025,—,—,None,None
10,RGxf,4F+,12.5,0.625,—,—,None,None


Verified 11 selected slabs; 253 parameter comparisons; all values, standard deviations, trace outputs and availability checks agree.


## 6. Complete actual-slab verification summary

The complete successful sets are found by full-cohort framework execution; **hand
calculations are performed for all selected examples**, not every slab in the cohort.
The audit below requires all 36 distinct pathways, one witness for every nonempty
success set, exactly ten distinct unfiltered draws per pathway, agreement for every
available nominal value and standard deviation, and matching availability/failure
predictions for all other steps. It checks that original dataset slabs remain unchanged.
The ten unfiltered draws are diagnostic examples, not a success-rate estimate; the
full-cohort counts above supply the actual pathway success rates.

In [374]:
# Assemble the final tables from the records collected by the 36 sections.
dataset_summary = pd.DataFrame(dataset_summary_records)
dataset_selections = pd.DataFrame(dataset_selection_records)
dataset_comparisons = pd.concat([
    case["comparison"].assign(pathway_number=case["pathway number"], target=case["target"])
    for case in dataset_case_records
], ignore_index=True)
# Require exactly one completed subsection for every inventory entry. A rerun of a
# section appends a second record, deliberately caught here; Run All resets the lists.
assert len(dataset_summary) == 36
assert dataset_summary["pathway number"].is_unique
assert set(dataset_summary["pathway number"]) == set(range(1, 37))
for target_name in [WEIGHT, ELASTICITY]:
    selected_paths = dataset_summary.loc[dataset_summary["target"] == target_name]
    assert set(selected_paths["pathway ID"]) == {path["id"] for path in inventory[target_name]}
    assert selected_paths["pathway ID"].is_unique
assert (dataset_summary["unfiltered draws"] == 10).all()
assert (dataset_summary["successful witness"] == (dataset_summary["successful set"] > 0).astype(int)).all()
for pathway_number, group in dataset_selections.groupby("pathway number"):
    random_draws = group.loc[group["selection"] != "successful witness", "dataset index"]
    assert len(random_draws) == 10 and random_draws.is_unique
assert len(dataset_selections) == 360 + int(dataset_summary["successful witness"].sum())
assert dataset_comparisons["passed"].all()
impossible_key = (ELASTICITY, ("kim_jamieson_table6", "bergfeld", "kochle"))
assert dataset_successful_indices[impossible_key] == []
# Source integrity includes observations, metadata, and all initially empty outputs.
assert [dataset_snapshot(slab) for slab in dataset_slabs] == dataset_before
display(dataset_summary.drop(columns="pathway ID").style.hide(axis="index").format(
    {"max |Δ value|": "{:.2e}", "max |Δ σ|": "{:.2e}"}, na_rep="—"))
print(f"All 36 pathways verified on {len(dataset_case_records)} selected slab/pathway cases.")
print(f"{(dataset_comparisons['status'] == 'PASS').sum():,} numeric comparisons and "
      f"{(dataset_comparisons['status'] == 'EXPECTED FAILURE').sum():,} matched unavailable outputs.")
print(f"Maximum nominal difference: {dataset_comparisons['|Δ value|'].max():.3e}; "
      f"maximum standard-deviation difference: {dataset_comparisons['|Δ σ|'].max():.3e}.")
print("Every selected graph output matched its trace; successful elasticity targets matched "
      "separate shear-weight executions; all source slabs unchanged.")

pathway number,target,density,E method,ν method,successful set,successful witness,unfiltered draws,unfiltered successes,numeric comparisons,matched unavailable,max |Δ value|,max |Δ σ|,verification
1,slab_weight_shear,data_flow,—,—,109,1,10,0,6,112,0.00e+00,3.55e-15,PASS
2,slab_weight_shear,geldsetzer,—,—,4187,1,10,4,91,39,0.00e+00,1.42e-14,PASS
3,slab_weight_shear,kim_jamieson_table2,—,—,5470,1,10,6,103,33,4.55e-13,2.84e-14,PASS
4,slab_weight_shear,kim_jamieson_table6,—,—,697,1,10,0,66,103,0.00e+00,7.11e-15,PASS
5,slab_weight_shear_with_elasticity,data_flow,bergfeld,kochle,6,1,10,0,24,177,0.00e+00,7.11e-15,PASS
6,slab_weight_shear_with_elasticity,data_flow,bergfeld,srivastava,8,1,10,0,11,150,0.00e+00,0.00e+00,PASS
7,slab_weight_shear_with_elasticity,data_flow,kochle,kochle,16,1,10,0,13,132,0.00e+00,3.55e-15,PASS
8,slab_weight_shear_with_elasticity,data_flow,kochle,srivastava,5,1,10,0,7,142,5.68e-14,2.13e-14,PASS
9,slab_weight_shear_with_elasticity,data_flow,wautier,kochle,17,1,10,0,45,208,5.68e-14,7.11e-15,PASS
10,slab_weight_shear_with_elasticity,data_flow,wautier,srivastava,12,1,10,0,7,194,5.68e-14,0.00e+00,PASS


All 36 pathways verified on 394 selected slab/pathway cases.
3,066 numeric comparisons and 4,704 matched unavailable outputs.
Maximum nominal difference: 4.547e-13; maximum standard-deviation difference: 2.842e-13.
Every selected graph output matched its trace; successful elasticity targets matched separate shear-weight executions; all source slabs unchanged.


## 7. Equation provenance and interpretation

Manual equations and coefficients are defined explicitly in this notebook rather
than imported from framework method functions. Their reference names correspond
to the implemented parameterizations documented in these local modules:

- [Density: Geldsetzer & Jamieson (2000); Kim & Jamieson (2014)](../src/snowpyt_mechparams/methods/layer/density.py)
- [Young's modulus: Bergfeld, Köchle, Wautier, and Schöttner](../src/snowpyt_mechparams/methods/layer/elastic_modulus.py)
- [Poisson's ratio: Köchle and Srivastava](../src/snowpyt_mechparams/methods/layer/poissons_ratio.py)
- [Weight integration, slope projection, and elasticity coverage target](../src/snowpyt_mechparams/methods/slab/coverage.py)

The framework scan evaluates every enumerated pathway on the complete ECTP cohort.
Independent hand calculations verify the selected actual slabs, including available
upstream calculations in failed pathways. These samples are not an exhaustive
test of every grain-specific coefficient or every domain boundary.